# NB99 — Analytic C₀ Derivation

**Goal:** Derive the period-0 cumulative sector ratio C₀ from the cascade ODE *without* numerical integration.

NB98 established that:
- C₀_Q = 6.6067, C₀_L = 5.9120 (converged ODE values, structural)
- amp_Q = C₀_Q / cp_trans_Q = 3.0836 (+0.24% vs √λ₊)
- amp_L = C₀_L / cp_trans_L = 4.8633 (−0.73% vs √24)
- The Gram bridge is approximate, κ-specific, and the error is structural

**Strategy:** The cascade ODE level-0 is a simple damped driven oscillator with exact solution. Each subsequent level receives driving from lower levels. Since all amplitudes are O(ρ) ≈ 0.069, we can chain analytic solutions through the cascade and compute C₀ analytically.

**Cascade ODE:**
```
Level 0:  dR₀/dt + κ·R₀ = ε·sin(ω·t)
Level k:  dRₖ/dt + κ·Rₖ = ε·sin(θₖ) − ε·sin(θₖ₋₁)/pₖ₋₁ + κ·Rₖ₋₁/pₖ₋₁
```
where θₖ = (ωt + Σⱼ₌₀ᵏ⁻¹ Rⱼ·Πⱼ) / Πₖ with Πₖ = p₀·p₁·…·pₖ₋₁.

In [1]:
# ── Setup ──
import sys, time
import numpy as np
from pathlib import Path

ROOT = Path.cwd().parent
if str(ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(ROOT / "scripts"))

from solenoid_algebra import (SA, RHO, KAPPA, EPSILON, OMEGA,
                               X4, X3, X2, LAM7, X4_LEP,
                               PHYSICAL_CROSSINGS, CP_PAIRS, SM_TARGETS,
                               ACTIVE_PRIMES)
from solenoid_system import SolenoidSystem
from solenoid_jax import integrate_all_branches_jax, warmup, detect_device

P4 = SA.P  # 210
primes = SA.primes  # [2, 3, 5, 7]
dlog = SA.dlog
kappa = KAPPA   # = epsilon = rho = 1/sqrt(210)
epsilon = EPSILON
omega = OMEGA   # = 2*pi

# Primorial products: Pi_0=1, Pi_1=2, Pi_2=6, Pi_3=30, Pi_4=210
Pi = [1]
for p in primes:
    Pi.append(Pi[-1] * p)
# Pi = [1, 2, 6, 30, 210]

# Gram matrix targets (from NB98)
s3 = np.sqrt(3)
M_gram = np.array([[9.0, s3], [s3, 3.0]])
lam_m, lam_p = np.linalg.eigvalsh(M_gram)
TARGET_AMP_Q = np.sqrt(lam_p)   # sqrt(6 + 2*sqrt(3))
TARGET_AMP_L = np.sqrt(24.0)    # sqrt(phi(35))

# Coprime crossings in period 0
coprime_cis = np.array([r for r in range(1, P4+1) if np.gcd(r, P4) == 1])
N_ci = len(coprime_cis)

# CRT sector labels
a3_lab = np.array([dlog[3][int(ci % 3)] for ci in coprime_cis])
a5_lab = np.array([dlog[5][int(ci % 5)] for ci in coprime_cis])
a7_lab = np.array([dlog[7][int(ci % 7)] for ci in coprime_cis])
sector_idx = a5_lab * 12 + a3_lab * 6 + a7_lab

# Physical sector indices (a5=0)
QUARK_G1  = 0*12 + 1*6 + 4  # 10
QUARK_G2  = 0*12 + 1*6 + 2  # 8
LEPTON_G1 = 0*12 + 0*6 + 1  # 1
LEPTON_G2 = 0*12 + 0*6 + 5  # 5

# Transient weights (algebraic, exact)
tw = {}
for a7 in range(6):
    cis = coprime_cis[a7_lab == a7].astype(float)
    tw[a7] = np.sum(np.exp(-2 * kappa * cis))
cp_trans_q = np.sqrt(tw[4] / tw[2])
cp_trans_l = np.sqrt(tw[1] / tw[5])

print(f"P4 = {P4}, primes = {list(primes)}")
print(f"kappa = epsilon = rho = {kappa:.8f}")
print(f"omega = {omega:.8f}")
print(f"Primorial products: {Pi}")
print(f"Period-0 crossings: {N_ci}")
print(f"\ncp_trans_Q = {cp_trans_q:.10f}")
print(f"cp_trans_L = {cp_trans_l:.10f}")
print(f"\nNB98 ODE targets: C0_Q = 6.6067422483, C0_L = 5.9119545825")
print(f"  amp_Q = 3.0836096405, amp_L = 4.8632881895")
print(f"\nGram targets: sqrt(lam+) = {TARGET_AMP_Q:.10f}, sqrt(24) = {TARGET_AMP_L:.10f}")

P4 = 210, primes = [2, 3, 5, 7]
kappa = epsilon = rho = 0.06900656
omega = 6.28318531
Primorial products: [1, 2, 6, 30, 210]
Period-0 crossings: 48

cp_trans_Q = 2.1425352164
cp_trans_L = 1.2156290872

NB98 ODE targets: C0_Q = 6.6067422483, C0_L = 5.9119545825
  amp_Q = 3.0836096405, amp_L = 4.8632881895

Gram targets: sqrt(lam+) = 3.0763780026, sqrt(24) = 4.8989794856


## S1: Zeroth-Order C₀ (Decoupled Exponential Decay)

At zeroth order (no inter-level coupling), each R_k decays independently:
$$R_k(t; \mathbf{j}) = 2\pi j_k \cdot e^{-\kappa t}$$

The branch-averaged wrapped R² at crossing ci, level lev is:
$$\langle R_{\text{lev}}^2 \rangle(ci) = \frac{1}{N_{\text{branches}}} \sum_{\mathbf{j}} [\text{wrap}(2\pi j_{\text{lev}} \cdot e^{-\kappa \cdot ci})]^2$$

Since only $j_{\text{lev}}$ matters for level lev (decoupled), this becomes an average over $j_{\text{lev}} \in \{0, \ldots, p_{\text{lev}}-1\}$.

In [2]:
# ── S1: Zeroth-order (decoupled) C₀ ──

def wrap(x):
    """Wrap to [-pi, pi]."""
    return np.mod(x + np.pi, 2*np.pi) - np.pi

# Branch-averaged R² at each crossing for each level (zeroth order)
# R_k(t; j) = 2*pi*j_k * exp(-kappa*t), decoupled
R_sq_zeroth = np.zeros((N_ci, 4))

ssys = SolenoidSystem()
all_branches = ssys.all_branches()
N_br = len(all_branches)  # 210

for ci_idx, ci in enumerate(coprime_cis):
    t = float(ci)
    decay = np.exp(-kappa * t)
    for lev in range(4):
        p_lev = primes[lev]
        # Average over all branches (only j_lev matters for this level)
        sum_sq = 0.0
        for br in all_branches:
            j_k = br[lev]
            R_raw = 2*np.pi * j_k * decay
            R_w = wrap(R_raw)
            sum_sq += R_w**2
        R_sq_zeroth[ci_idx, lev] = sum_sq / N_br

# Accumulate into sectors (exactly as NB98)
cum_sq_z = np.zeros((48, N_ci, 4))
cum_ct = np.zeros((48, N_ci), dtype=np.int64)
for j in range(N_ci):
    if j > 0:
        cum_sq_z[:, j, :] = cum_sq_z[:, j-1, :]
        cum_ct[:, j] = cum_ct[:, j-1]
    s = sector_idx[j]
    cum_sq_z[s, j, :] += R_sq_zeroth[j, :]
    cum_ct[s, j] += 1

# Extract C0 at R4 level
j_end = N_ci - 1
lev = 3  # R4

g1_q_z = cum_sq_z[QUARK_G1, j_end, lev] / max(cum_ct[QUARK_G1, j_end], 1)
g2_q_z = cum_sq_z[QUARK_G2, j_end, lev] / max(cum_ct[QUARK_G2, j_end], 1)
C0_Q_z = np.sqrt(g1_q_z / g2_q_z)

g1_l_z = cum_sq_z[LEPTON_G1, j_end, lev] / max(cum_ct[LEPTON_G1, j_end], 1)
g2_l_z = cum_sq_z[LEPTON_G2, j_end, lev] / max(cum_ct[LEPTON_G2, j_end], 1)
C0_L_z = np.sqrt(g1_l_z / g2_l_z)

amp_Q_z = C0_Q_z / cp_trans_q
amp_L_z = C0_L_z / cp_trans_l

ODE_C0_Q = 6.6067422483
ODE_C0_L = 5.9119545825

print("=" * 65)
print("S1: ZEROTH-ORDER (DECOUPLED) C₀")
print("=" * 65)
print(f"\n{'':>12} {'C0_zeroth':>12} {'C0_ODE':>12} {'ratio':>10} {'amp':>10}")
print("-" * 60)
print(f"{'QUARK':>12} {C0_Q_z:12.6f} {ODE_C0_Q:12.6f} {C0_Q_z/ODE_C0_Q:10.4f} {amp_Q_z:10.4f}")
print(f"{'LEPTON':>12} {C0_L_z:12.6f} {ODE_C0_L:12.6f} {C0_L_z/ODE_C0_L:10.4f} {amp_L_z:10.4f}")

print(f"\nZeroth-order amplification: Q = {amp_Q_z:.6f}, L = {amp_L_z:.6f}")
print(f"ODE amplification:          Q = 3.083610,     L = 4.863288")
print(f"\nZeroth-order accounts for: Q = {C0_Q_z/ODE_C0_Q*100:.1f}%, L = {C0_L_z/ODE_C0_L*100:.1f}%")
print(f"Coupling contribution:      Q = {(1 - C0_Q_z/ODE_C0_Q)*100:.1f}%, L = {(1 - C0_L_z/ODE_C0_L)*100:.1f}%")

S1: ZEROTH-ORDER (DECOUPLED) C₀

                C0_zeroth       C0_ODE      ratio        amp
------------------------------------------------------------
       QUARK 41393.659263     6.606742  6265.3662 19319.9435
      LEPTON     5.781378     5.911955     0.9779     4.7559

Zeroth-order amplification: Q = 19319.943470, L = 4.755874
ODE amplification:          Q = 3.083610,     L = 4.863288

Zeroth-order accounts for: Q = 626536.6%, L = 97.8%
Coupling contribution:      Q = -626436.6%, L = 2.2%


## S2: Level-0 Exact Analytic Solution

Level 0 of the cascade is an exactly solvable damped driven oscillator:
$$\frac{dR_0}{dt} + \kappa R_0 = \varepsilon \sin(\omega t)$$

with $R_0(0) = 2\pi j_0$. The solution is:
$$R_0(t; j_0) = 2\pi j_0 \cdot e^{-\kappa t} + R_0^{(\text{ss})}(t)$$

where the steady-state (particular) solution is:
$$R_0^{(\text{ss})}(t) = \frac{\varepsilon}{\kappa^2 + \omega^2}\left(\kappa \sin\omega t - \omega \cos\omega t\right)$$

We verify this against the ODE integrator then use it to drive level 1.

In [4]:
# ── S2: Level-0 exact analytic solution ──

# Steady-state coefficients for dR0/dt + kappa*R0 = epsilon*sin(omega*t)
# Particular solution: R0_ss(t) = A0*sin(wt) + B0*cos(wt)
denom0 = kappa**2 + omega**2
A0_ss = epsilon * kappa / denom0   # sin coefficient
B0_ss = -epsilon * omega / denom0  # cos coefficient
amp0_ss = epsilon / np.sqrt(denom0)

# Full solution: R0(t;j0) = C_j*exp(-kappa*t) + A0*sin(wt) + B0*cos(wt)
# IC: R0(0) = 2*pi*j0 = C_j + B0  =>  C_j = 2*pi*j0 - B0

def R0_analytic(t, j0):
    """Exact R_0 for branch index j0."""
    C = 2*np.pi * j0 - B0_ss
    return C * np.exp(-kappa * t) + A0_ss * np.sin(omega * t) + B0_ss * np.cos(omega * t)

print("Level-0 steady-state:")
print(f"  A0 = eps*kappa/(kappa^2+omega^2) = {A0_ss:.8e}")
print(f"  B0 = -eps*omega/(kappa^2+omega^2) = {B0_ss:.8e}")
print(f"  Amplitude = eps/sqrt(kappa^2+omega^2) = {amp0_ss:.8e}")
print(f"  IC correction: C(j0) = 2*pi*j0 - B0 = 2*pi*j0 + {-B0_ss:.8e}")

# Verify against ODE integration (single branch, j=(1,0,0,0))
warmup()
test_branch = (1, 0, 0, 0)
t_test = coprime_cis.astype(float)
res_test = ssys.integrate_all_branches(
    [test_branch], t_test, float(P4+1), backend='jax'
)
R_ode_br = res_test[test_branch]  # (N_ci, 4)
R0_ode = R_ode_br[:, 0]  # level 0

R0_ana = np.array([R0_analytic(float(ci), 1) for ci in coprime_cis])

# Compare
max_err = np.max(np.abs(R0_ana - R0_ode))
rel_err = max_err / np.max(np.abs(R0_ode))
print(f"\nVerification (branch j=(1,0,0,0), level 0):")
print(f"  Max absolute error: {max_err:.4e}")
print(f"  Max relative error: {rel_err:.4e}")
if rel_err > 0:
    print(f"  R0_analytic matches ODE to {-np.log10(rel_err):.1f} digits")
else:
    print(f"  R0_analytic matches ODE exactly")

# Show a few values
print(f"\n{'ci':>6} {'R0_analytic':>14} {'R0_ODE':>14} {'diff':>12}")
print("-" * 50)
for idx in [0, 5, 10, 20, 30, 47]:
    ci = coprime_cis[idx]
    print(f"{ci:6d} {R0_ana[idx]:14.8f} {R0_ode[idx]:14.8f} {R0_ana[idx]-R0_ode[idx]:12.4e}")

Level-0 steady-state:
  A0 = eps*kappa/(kappa^2+omega^2) = 1.20605909e-04
  B0 = -eps*omega/(kappa^2+omega^2) = -1.09814099e-02
  Amplitude = eps/sqrt(kappa^2+omega^2) = 1.09820722e-02
  IC correction: C(j0) = 2*pi*j0 - B0 = 2*pi*j0 + 1.09814099e-02
  JAX [CPU (1 device(s))]: 1 branches, 48 eval pts, T=211.0 — 0.66s

Verification (branch j=(1,0,0,0), level 0):
  Max absolute error: 9.8943e-12
  Max relative error: 1.6874e-12
  R0_analytic matches ODE to 11.8 digits

    ci    R0_analytic         R0_ODE         diff
--------------------------------------------------
     1     5.86349380     5.86349380  -4.4169e-12
    23     1.27622269     1.27622269  -7.0211e-12
    43     0.31280908     0.31280908   1.1896e-13
    89     0.00256058     0.00256058  -2.2913e-13
   137    -0.01048805    -0.01048805  -5.8389e-14
   209    -0.01097798    -0.01097798  -2.5258e-13


## S3: Full Cascade — Numerical Solution with Analytic Verification

The cascade equations for k≥1 involve sin(θ_k) where θ_k depends nonlinearly on lower R values. For a fully analytic result we'd need to linearize, but first let's verify we can reproduce the ODE C₀ with a different approach.

**Key insight from S1:** The zeroth-order (decoupled) model gives C₀_L ≈ 97.8% of the ODE value, but C₀_Q diverges because the QUARK g2 sector (ci=191) has zero transient. The driven response from inter-level coupling is what fills in the late-time crossings.

**Approach:** Rather than solving perturbatively (which requires handling nested nonlinearities), we'll extract the amplification structure by decomposing the ODE solution into:
1. **Transient component**: depends on branch j, decays as exp(−κt)  
2. **Driven component**: same for all branches, periodic in t  
3. **Cross terms**: interference between transient and driven

In [5]:
# ── S3: ODE reference integration and transient/driven decomposition ──

# Integrate ALL 210 branches at period-0 crossings
t0 = time.time()
res_all = integrate_all_branches_jax(
    all_branches, t_test, float(P4+1),
    primes=list(primes), omega=omega, epsilon=epsilon, kappa=kappa,
    rtol=1e-12, atol=1e-14, verbose=False
)
dt = time.time() - t0
print(f"Integrated 210 branches x 48 crossings in {dt:.1f}s")

# Stack: (210, 48, 4)
R_all = np.stack([res_all[br][:N_ci] for br in sorted(res_all.keys())], axis=0)
R_wrapped = np.mod(R_all + np.pi, 2*np.pi) - np.pi

# The DRIVEN component is what you get when the transient has decayed.
# A clean way: integrate a single "zero-IC" system where all j_k = 0.
# That branch has R_k(0) = 0, so no transient — pure driven response.
zero_branch = (0, 0, 0, 0)
R_driven_raw = res_all[zero_branch][:N_ci]  # (48, 4) — same for all branches
R_driven = np.mod(R_driven_raw + np.pi, 2*np.pi) - np.pi

# The TRANSIENT is the difference: R_total - R_driven
# For each branch: R_trans(t;j) = R_total(t;j) - R_driven(t)
R_trans = R_all - R_driven_raw[np.newaxis, :, :]  # (210, 48, 4) raw difference
R_trans_w = np.mod(R_trans + np.pi, 2*np.pi) - np.pi

print(f"\nDriven response amplitude (RMS across crossings, level R4):")
print(f"  |R_driven| at ci=1:   {np.abs(R_driven[0, 3]):.6e}")
print(f"  |R_driven| at ci=11:  {np.abs(R_driven[np.where(coprime_cis==11)[0][0], 3]):.6e}")
print(f"  |R_driven| at ci=191: {np.abs(R_driven[np.where(coprime_cis==191)[0][0], 3]):.6e}")

# Branch-averaged R² components at each crossing (level R4 = lev 3)
lev = 3
R_sq_avg = (R_wrapped**2).mean(axis=0)  # (48, 4)
R_sq_driven = R_driven**2  # (48, 4) no averaging needed — same for all branches
R_sq_trans = (R_trans_w**2).mean(axis=0)  # (48, 4)

# Cross-term: <R²> = <R_trans²> + R_driven² + 2*<R_trans*R_driven>
# => cross = <R²> - <R_trans²> - R_driven²
R_sq_cross = R_sq_avg - R_sq_trans - R_sq_driven

# Show decomposition at physical crossings
print(f"\n{'ci':>6} {'R²_total':>12} {'R²_trans':>12} {'R²_driven':>12} {'R²_cross':>12} {'trans%':>8}")
print("-" * 70)
for ci_val in [11, 31, 61, 191]:
    idx = np.where(coprime_cis == ci_val)[0][0]
    tot = R_sq_avg[idx, lev]
    tra = R_sq_trans[idx, lev]
    dri = R_sq_driven[idx, lev]
    cro = R_sq_cross[idx, lev]
    pct = tra/tot*100 if tot > 0 else 0
    print(f"{ci_val:6d} {tot:12.6e} {tra:12.6e} {dri:12.6e} {cro:12.6e} {pct:7.1f}%")

Integrated 210 branches x 48 crossings in 1.6s

Driven response amplitude (RMS across crossings, level R4):
  |R_driven| at ci=1:   7.425265e-04
  |R_driven| at ci=11:  4.181629e-01
  |R_driven| at ci=191: 2.792372e-01

    ci     R²_total     R²_trans    R²_driven     R²_cross   trans%
----------------------------------------------------------------------
    11 3.409538e+00 3.452753e+00 1.748602e-01 -2.180753e-01   101.3%
    31 3.895099e+00 3.925382e+00 5.254390e-02 -8.282719e-02   100.8%
    61 1.114439e-01 3.088476e-01 6.809376e-02 -2.654975e-01   277.1%
   191 7.811256e-02 7.440121e-08 7.797341e-02 1.390695e-04     0.0%


In [6]:
# ── S3b: C₀ anatomy — contribution from each component ──

# Build cumulative sector sums for total, transient, driven, cross
def sector_C0(R_sq_vals):
    """Compute C₀ from per-crossing branch-avg R² values (shape N_ci x 4)."""
    cum = np.zeros((48, N_ci, 4))
    ct = np.zeros((48, N_ci), dtype=np.int64)
    for j in range(N_ci):
        if j > 0:
            cum[:, j, :] = cum[:, j-1, :]
            ct[:, j] = ct[:, j-1]
        s = sector_idx[j]
        cum[s, j, :] += R_sq_vals[j, :]
        ct[s, j] += 1
    j0 = N_ci - 1
    result = {}
    for name, (s1, s2) in [('QUARK', (QUARK_G1, QUARK_G2)),
                             ('LEPTON', (LEPTON_G1, LEPTON_G2))]:
        g1 = cum[s1, j0, lev] / max(ct[s1, j0], 1)
        g2 = cum[s2, j0, lev] / max(ct[s2, j0], 1)
        result[name] = {'g1': g1, 'g2': g2, 'C0': np.sqrt(g1/g2) if g2 > 0 else np.inf}
    return result

res_total = sector_C0(R_sq_avg)
res_trans = sector_C0(R_sq_trans)
res_driv  = sector_C0(R_sq_driven)
res_cross = sector_C0(R_sq_cross)

print("=" * 70)
print("S3b: C₀ ANATOMY — per-component sector ratios at R4")
print("=" * 70)
print(f"\n{'Component':>12} {'Q_g1(ci=11)':>14} {'Q_g2(ci=191)':>14} {'C0_Q':>10}")
print("-" * 55)
for lbl, res in [('TOTAL', res_total), ('TRANSIENT', res_trans),
                  ('DRIVEN', res_driv), ('CROSS', res_cross)]:
    q = res['QUARK']
    print(f"{lbl:>12} {q['g1']:14.6e} {q['g2']:14.6e} {q['C0']:10.4f}")

print(f"\n{'Component':>12} {'L_g1(ci=31)':>14} {'L_g2(ci=61)':>14} {'C0_L':>10}")
print("-" * 55)
for lbl, res in [('TOTAL', res_total), ('TRANSIENT', res_trans),
                  ('DRIVEN', res_driv), ('CROSS', res_cross)]:
    l = res['LEPTON']
    print(f"{lbl:>12} {l['g1']:14.6e} {l['g2']:14.6e} {l['C0']:10.4f}")

# Compare to NB98 targets
print(f"\nODE targets: C0_Q = {ODE_C0_Q:.6f}, C0_L = {ODE_C0_L:.6f}")
print(f"Total match: C0_Q = {res_total['QUARK']['C0']:.6f}, C0_L = {res_total['LEPTON']['C0']:.6f}")

# The KEY: what is the ratio of g1/g2 for each component?
print(f"\n--- Component ratios (g1/g2) ---")
for name in ['QUARK', 'LEPTON']:
    print(f"\n  {name}:")
    for lbl, res in [('total', res_total), ('trans', res_trans),
                      ('driven', res_driv), ('cross', res_cross)]:
        r = res[name]
        ratio_sq = r['g1'] / r['g2'] if r['g2'] != 0 else float('inf')
        print(f"    {lbl:>8}: g1/g2 = {ratio_sq:12.4f} (C0 = {np.sqrt(abs(ratio_sq)):8.4f})")

S3b: C₀ ANATOMY — per-component sector ratios at R4

   Component    Q_g1(ci=11)   Q_g2(ci=191)       C0_Q
-------------------------------------------------------
       TOTAL   3.409538e+00   7.811256e-02     6.6067
   TRANSIENT   3.452753e+00   7.440121e-08  6812.2847
      DRIVEN   1.748602e-01   7.797341e-02     1.4975
       CROSS  -2.180753e-01   1.390695e-04        nan

   Component    L_g1(ci=31)    L_g2(ci=61)       C0_L
-------------------------------------------------------
       TOTAL   3.895099e+00   1.114439e-01     5.9120
   TRANSIENT   3.925382e+00   3.088476e-01     3.5651
      DRIVEN   5.254390e-02   6.809376e-02     0.8784
       CROSS  -8.282719e-02  -2.654975e-01        inf

ODE targets: C0_Q = 6.606742, C0_L = 5.911955
Total match: C0_Q = 6.606742, C0_L = 5.911955

--- Component ratios (g1/g2) ---

  QUARK:
       total: g1/g2 =      43.6490 (C0 =   6.6067)
       trans: g1/g2 = 46407222.6627 (C0 = 6812.2847)
      driven: g1/g2 =       2.2426 (C0 =   1.4975)
  

C:\Users\mlf\AppData\Local\Temp\ipykernel_45384\4226470361.py:21: RuntimeWarning: invalid value encountered in sqrt
  result[name] = {'g1': g1, 'g2': g2, 'C0': np.sqrt(g1/g2) if g2 > 0 else np.inf}


In [7]:
# ── S3c: Driven response structure ──

# The driven R_k(t) = R_k(t; j=0000) — same for all branches.
# This is the cascade-generated signal. Let's look at its anatomy.

# Dense time grid for frequency analysis
from scipy.fft import fft, fftfreq

# We already have R_driven at the 48 coprime crossings.
# For Fourier analysis, we need uniformly-spaced samples.
# Integrate zero-branch at fine uniform spacing in [0, P4].
from scipy.integrate import solve_ivp

R0_driven_ic = np.zeros(4)
sol = solve_ivp(
    ssys.cascade_ode, [0, float(P4)], R0_driven_ic,
    t_eval=np.linspace(0, float(P4), 4096),
    method='DOP853', rtol=1e-12, atol=1e-14
)

print(f"Driven cascade solution: {sol.t.shape[0]} time points in [0, {P4}]")
print(f"Final R values: {sol.y[:, -1]}")

# Fourier transform of R_3(t) driven
R3d = sol.y[3, :]
N_fft = len(R3d)
dt_fft = float(P4) / N_fft
freqs = fftfreq(N_fft, dt_fft)
spectrum = np.abs(fft(R3d)) / N_fft

# Show dominant frequencies
pos_mask = freqs > 0
freqs_pos = freqs[pos_mask]
spec_pos = spectrum[pos_mask]
top_idx = np.argsort(spec_pos)[::-1][:15]

print(f"\nTop 15 frequencies in driven R_3(t):")
print(f"{'freq':>10} {'period':>10} {'amplitude':>12} {'ratio_to_w/210':>16}")
print("-" * 52)
for idx in top_idx:
    f = freqs_pos[idx]
    T = 1/f if f > 0 else np.inf
    # Express as ratio to omega/(2*pi*210) = 1/210
    ratio = f * P4
    print(f"{f:10.4f} {T:10.2f} {spec_pos[idx]:12.6e} {ratio:16.2f}")

# Key solenoid frequencies: omega/Pi_k = 2*pi/Pi_k
# Natural periods: Pi_k (= 1, 2, 6, 30, 210)
print(f"\nSolenoid natural frequencies (omega/Pi_k = 2*pi/Pi_k):")
for k in range(5):
    f_k = 1.0 / Pi[k]
    print(f"  Level {k}: freq = {f_k:.6f} (period = {Pi[k]})")

Driven cascade solution: 4096 time points in [0, 210]
Final R values: [-0.0109814  -0.01646765 -0.05830552 -0.28431159]

Top 15 frequencies in driven R_3(t):
      freq     period    amplitude   ratio_to_w/210
----------------------------------------------------
    0.0333      30.00 1.566987e-01             7.00
    0.0048     210.00 1.871097e-02             1.00
    0.0095     105.00 1.521794e-02             2.00
    0.0143      70.00 1.218853e-02             3.00
    0.0190      52.50 9.966256e-03             4.00
    0.0238      42.00 8.374323e-03             5.00
    0.0286      35.00 7.220900e-03             6.00
    0.1667       6.00 6.597907e-03            35.00
    0.0381      26.25 5.558452e-03             8.00
    0.0429      23.33 4.938636e-03             9.00
    0.0476      21.00 4.446832e-03            10.00
    0.0524      19.09 4.047586e-03            11.00
    0.0571      17.50 3.715694e-03            12.00
    0.0619      16.15 3.434692e-03            13.00
    0.066

## S4: First-Order Perturbative Cascade

At first order in ε=κ=ρ, each level receives driving at its natural frequency minus the subharmonic from below. The sin(θₖ) terms linearize as sin(ωt/Πₖ) since the R-corrections are O(ρ²):

| Level | First-order driving |
|-------|-------------------|
| 0 | ε·sin(ωt) |
| 1 | ε·sin(ωt/2) − (ε/2)·sin(ωt) |
| 2 | ε·sin(ωt/6) − (ε/3)·sin(ωt/2) |
| 3 | ε·sin(ωt/30) − (ε/5)·sin(ωt/6) |

Each is dRₖ/dt + κ·Rₖ = Σ aₘ·sin(Ωₘt) with exact solution.

In [8]:
# ── S4: First-order perturbative cascade (driven response, IC=0) ──

def damped_driven_solution(driving_terms, kap, t_arr):
    """
    Solve dR/dt + kap*R = sum_m a_m*sin(Om_m*t) with R(0) = 0.
    
    driving_terms: list of (amplitude, frequency) pairs
    Returns R(t) for each t in t_arr.
    """
    R = np.zeros_like(t_arr)
    B_total = 0.0  # sum of cos-coefficients at t=0
    for a_m, Om_m in driving_terms:
        denom = kap**2 + Om_m**2
        A_m = a_m * kap / denom    # sin coeff
        B_m = -a_m * Om_m / denom  # cos coeff
        R += A_m * np.sin(Om_m * t_arr) + B_m * np.cos(Om_m * t_arr)
        B_total += B_m
    # IC correction: R(0) = C + B_total = 0 => C = -B_total
    R += (-B_total) * np.exp(-kap * t_arr)
    return R

t_arr = coprime_cis.astype(float)

# Level 0: exact (already have R0_analytic for j0=0)
R0_pert = np.array([R0_analytic(t, 0) for t in t_arr])

# Level 1: eps*sin(wt/2) - (eps/2)*sin(wt)
R1_pert = damped_driven_solution(
    [(epsilon, omega/2), (-epsilon/2, omega)],
    kappa, t_arr
)

# Level 2: eps*sin(wt/6) - (eps/3)*sin(wt/2)
R2_pert = damped_driven_solution(
    [(epsilon, omega/6), (-epsilon/3, omega/2)],
    kappa, t_arr
)

# Level 3: eps*sin(wt/30) - (eps/5)*sin(wt/6)
R3_pert = damped_driven_solution(
    [(epsilon, omega/30), (-epsilon/5, omega/6)],
    kappa, t_arr
)

# Compare to ODE driven response (j=0000 branch)
R_driv_ode = res_all[zero_branch][:N_ci]  # (48, 4)

print("=" * 70)
print("S4: FIRST-ORDER PERTURBATIVE vs ODE (driven response, IC=0 branch)")
print("=" * 70)
for lev_name, R_pert, lev_idx in [('R0', R0_pert, 0), ('R1', R1_pert, 1),
                                    ('R2', R2_pert, 2), ('R3', R3_pert, 3)]:
    R_ode = R_driv_ode[:, lev_idx]
    max_abs = np.max(np.abs(R_pert - R_ode))
    max_sig = np.max(np.abs(R_ode))
    rel = max_abs / max_sig if max_sig > 0 else 0
    print(f"  {lev_name}: max|pert-ODE| = {max_abs:.4e}, max|ODE| = {max_sig:.4e}, rel = {rel:.2%}")

# Detailed comparison for R3
print(f"\n{'ci':>6} {'R3_pert':>14} {'R3_ODE':>14} {'diff':>12} {'rel%':>8}")
print("-" * 58)
for idx in [0, 3, 5, 10, 20, 30, 40, 47]:
    ci = coprime_cis[idx]
    rp = R3_pert[idx]
    ro = R_driv_ode[idx, 3]
    d = rp - ro
    r = d/ro*100 if ro != 0 else 0
    print(f"{ci:6d} {rp:14.8f} {ro:14.8f} {d:12.4e} {r:7.2f}%")

# Steady-state amplitudes (late crossings)
print(f"\nSteady-state amplitudes (last 10 crossings RMS):")
for lev_name, R_pert, lev_idx in [('R0', R0_pert, 0), ('R1', R1_pert, 1),
                                    ('R2', R2_pert, 2), ('R3', R3_pert, 3)]:
    rms_pert = np.sqrt(np.mean(R_pert[-10:]**2))
    rms_ode = np.sqrt(np.mean(R_driv_ode[-10:, lev_idx]**2))
    print(f"  {lev_name}: pert = {rms_pert:.6f}, ODE = {rms_ode:.6f}, ratio = {rms_pert/rms_ode:.4f}")

S4: FIRST-ORDER PERTURBATIVE vs ODE (driven response, IC=0 branch)
  R0: max|pert-ODE| = 2.0040e-13, max|ODE| = 1.0981e-02, rel = 0.00%
  R1: max|pert-ODE| = 2.0104e-03, max|ODE| = 4.3169e-02, rel = 4.66%
  R2: max|pert-ODE| = 2.4799e-03, max|ODE| = 4.3805e-02, rel = 5.66%
  R3: max|pert-ODE| = 6.2170e-03, max|ODE| = 4.3814e-01, rel = 1.42%

    ci        R3_pert         R3_ODE         diff     rel%
----------------------------------------------------------
     1     0.00060191     0.00074253  -1.4061e-04  -18.94%
    17     0.32689980     0.33072223  -3.8224e-03   -1.16%
    23    -0.06304926    -0.05932556  -3.7237e-03    6.28%
    43     0.33177860     0.33398172  -2.2031e-03   -0.66%
    89    -0.30316081    -0.30158035  -1.5805e-03    0.52%
   137     0.23902249     0.23900291   1.9586e-05    0.01%
   179    -0.30377081    -0.30241501  -1.3558e-03    0.45%
   209    -0.30377188    -0.30241730  -1.3546e-03    0.45%

Steady-state amplitudes (last 10 crossings RMS):
  R0: pert = 0.0

In [9]:
# ── S4b: First-order analytic C₀ ──
# At first order: R_k(t;j) = R_k^driv(t) + 2*pi*j_k*exp(-kappa*t)
# The branch transient is independent of the driving at O(rho).
# R² depends on the wrapping, which mixes transient and driven.

R_pert_driven = np.column_stack([R0_pert, R1_pert, R2_pert, R3_pert])  # (48, 4)

# Compute branch-averaged R² at each crossing
R_sq_pert = np.zeros((N_ci, 4))
for ci_idx, ci in enumerate(coprime_cis):
    t = float(ci)
    decay = np.exp(-kappa * t)
    for lev in range(4):
        p_lev = primes[lev]
        R_driv_val = R_pert_driven[ci_idx, lev]
        sq_sum = 0.0
        for j_k in range(p_lev):
            R_total = R_driv_val + 2*np.pi * j_k * decay
            R_w = wrap(R_total)
            sq_sum += R_w**2
        # Average over all branches: each j_k appears in P4/p_lev branches
        R_sq_pert[ci_idx, lev] = sq_sum / p_lev

# Accumulate into sectors
cum_sq_p = np.zeros((48, N_ci, 4))
cum_ct_p = np.zeros((48, N_ci), dtype=np.int64)
for j in range(N_ci):
    if j > 0:
        cum_sq_p[:, j, :] = cum_sq_p[:, j-1, :]
        cum_ct_p[:, j] = cum_ct_p[:, j-1]
    s = sector_idx[j]
    cum_sq_p[s, j, :] += R_sq_pert[j, :]
    cum_ct_p[s, j] += 1

# Extract C0 at R4 (lev=3)
j_end = N_ci - 1
lev = 3

g1_q_p = cum_sq_p[QUARK_G1, j_end, lev] / max(cum_ct_p[QUARK_G1, j_end], 1)
g2_q_p = cum_sq_p[QUARK_G2, j_end, lev] / max(cum_ct_p[QUARK_G2, j_end], 1)
C0_Q_p = np.sqrt(g1_q_p / g2_q_p)

g1_l_p = cum_sq_p[LEPTON_G1, j_end, lev] / max(cum_ct_p[LEPTON_G1, j_end], 1)
g2_l_p = cum_sq_p[LEPTON_G2, j_end, lev] / max(cum_ct_p[LEPTON_G2, j_end], 1)
C0_L_p = np.sqrt(g1_l_p / g2_l_p)

amp_Q_p = C0_Q_p / cp_trans_q
amp_L_p = C0_L_p / cp_trans_l

print("=" * 70)
print("S4b: FIRST-ORDER PERTURBATIVE C₀")
print("=" * 70)
print(f"\n{'':>14} {'C0_pert':>12} {'C0_ODE':>12} {'err%':>10} {'amp':>10}")
print("-" * 60)
print(f"{'QUARK':>14} {C0_Q_p:12.6f} {ODE_C0_Q:12.6f} {(C0_Q_p/ODE_C0_Q-1)*100:9.3f}% {amp_Q_p:10.6f}")
print(f"{'LEPTON':>14} {C0_L_p:12.6f} {ODE_C0_L:12.6f} {(C0_L_p/ODE_C0_L-1)*100:9.3f}% {amp_L_p:10.6f}")

print(f"\nODE amplification:  Q = 3.083610, L = 4.863288")
print(f"Pert amplification: Q = {amp_Q_p:.6f}, L = {amp_L_p:.6f}")
print(f"Gram targets:       sqrt(lam+) = {TARGET_AMP_Q:.6f}, sqrt(24) = {TARGET_AMP_L:.6f}")

# Per-crossing comparison (R4 level)
print(f"\n{'ci':>6} {'R²_pert':>12} {'R²_ODE':>12} {'diff%':>8}")
print("-" * 42)
for ci_val in [11, 31, 61, 191]:
    idx = np.where(coprime_cis == ci_val)[0][0]
    rp = R_sq_pert[idx, lev]
    ro = R_sq_avg[idx, lev]
    d = (rp/ro - 1)*100 if ro > 0 else 0
    print(f"{ci_val:6d} {rp:12.6e} {ro:12.6e} {d:7.2f}%")

S4b: FIRST-ORDER PERTURBATIVE C₀

                    C0_pert       C0_ODE       err%        amp
------------------------------------------------------------
         QUARK     6.740699     6.606742     2.028%   3.146132
        LEPTON    10.167800     5.911955    71.987%   8.364229

ODE amplification:  Q = 3.083610, L = 4.863288
Pert amplification: Q = 3.146132, L = 8.364229
Gram targets:       sqrt(lam+) = 3.076378, sqrt(24) = 4.898979

    ci      R²_pert       R²_ODE    diff%
------------------------------------------
    11 3.536793e+00 3.409538e+00    3.73%
    31 3.642828e+00 3.895099e+00   -6.48%
    61 3.523585e-02 1.114439e-01  -68.38%
   191 7.783947e-02 7.811256e-02   -0.35%


In [10]:
# ── S4c: Hybrid approach — ODE driven + analytic transient ──
# At first order: R_k(t;j) = R_k^driv(t) + 2*pi*j_k*exp(-kappa*t)
# Use EXACT ODE for R^driv (from j=0000 branch), analytic exp for transient

R_driv_exact = res_all[zero_branch][:N_ci]  # (48, 4) from ODE

R_sq_hybrid = np.zeros((N_ci, 4))
for ci_idx, ci in enumerate(coprime_cis):
    t = float(ci)
    decay = np.exp(-kappa * t)
    for lev in range(4):
        p_lev = primes[lev]
        R_driv_val = R_driv_exact[ci_idx, lev]
        sq_sum = 0.0
        for j_k in range(p_lev):
            R_total = R_driv_val + 2*np.pi * j_k * decay
            R_w = wrap(R_total)
            sq_sum += R_w**2
        R_sq_hybrid[ci_idx, lev] = sq_sum / p_lev

# Accumulate into sectors
cum_sq_h = np.zeros((48, N_ci, 4))
cum_ct_h = np.zeros((48, N_ci), dtype=np.int64)
for j in range(N_ci):
    if j > 0:
        cum_sq_h[:, j, :] = cum_sq_h[:, j-1, :]
        cum_ct_h[:, j] = cum_ct_h[:, j-1]
    s = sector_idx[j]
    cum_sq_h[s, j, :] += R_sq_hybrid[j, :]
    cum_ct_h[s, j] += 1

# C₀ at R4
j_end = N_ci - 1
lev = 3

g1_q_h = cum_sq_h[QUARK_G1, j_end, lev] / max(cum_ct_h[QUARK_G1, j_end], 1)
g2_q_h = cum_sq_h[QUARK_G2, j_end, lev] / max(cum_ct_h[QUARK_G2, j_end], 1)
C0_Q_h = np.sqrt(g1_q_h / g2_q_h)

g1_l_h = cum_sq_h[LEPTON_G1, j_end, lev] / max(cum_ct_h[LEPTON_G1, j_end], 1)
g2_l_h = cum_sq_h[LEPTON_G2, j_end, lev] / max(cum_ct_h[LEPTON_G2, j_end], 1)
C0_L_h = np.sqrt(g1_l_h / g2_l_h)

amp_Q_h = C0_Q_h / cp_trans_q
amp_L_h = C0_L_h / cp_trans_l

print("=" * 70)
print("S4c: HYBRID C₀ (ODE driven + analytic transient 2*pi*j*exp(-kt))")
print("=" * 70)
print(f"\n{'':>14} {'C0_hybrid':>12} {'C0_ODE':>12} {'err%':>10} {'amp':>10}")
print("-" * 60)
print(f"{'QUARK':>14} {C0_Q_h:12.6f} {ODE_C0_Q:12.6f} {(C0_Q_h/ODE_C0_Q-1)*100:9.4f}% {amp_Q_h:10.6f}")
print(f"{'LEPTON':>14} {C0_L_h:12.6f} {ODE_C0_L:12.6f} {(C0_L_h/ODE_C0_L-1)*100:9.4f}% {amp_L_h:10.6f}")

# Per-crossing
print(f"\n{'ci':>6} {'R²_hyb':>12} {'R²_ODE':>12} {'diff%':>10}")
print("-" * 45)
for ci_val in [1, 11, 31, 61, 191]:
    idx = np.where(coprime_cis == ci_val)[0][0]
    rh = R_sq_hybrid[idx, lev]
    ro = R_sq_avg[idx, lev]
    d = (rh/ro - 1)*100 if ro > 0 else 0
    print(f"{ci_val:6d} {rh:12.6e} {ro:12.6e} {d:9.4f}%")

# INTERPRETATION: How well does the decomposition
# R(t;j) = R^driv(t) + 2*pi*j*exp(-kappa*t) work?
# If it works perfectly, the hybrid C₀ = ODE C₀.
print(f"\nINTERPRETATION:")
print(f"  If R(t;j) = R_driv(t) + 2pi*j*exp(-kt) is exact,")
print(f"  then hybrid C₀ = ODE C₀. Deviation measures nonlinear coupling.")

S4c: HYBRID C₀ (ODE driven + analytic transient 2*pi*j*exp(-kt))

                  C0_hybrid       C0_ODE       err%        amp
------------------------------------------------------------
         QUARK     6.737156     6.606742    1.9739%   3.144478
        LEPTON    10.172277     5.911955   72.0628%   8.367912

    ci       R²_hyb       R²_ODE      diff%
---------------------------------------------
     1 2.279983e+00 1.905024e+00   19.6827%
    11 3.540058e+00 3.409538e+00    3.8281%
    31 3.643555e+00 3.895099e+00   -6.4580%
    61 3.521186e-02 1.114439e-01  -68.4040%
   191 7.799328e-02 7.811256e-02   -0.1527%

INTERPRETATION:
  If R(t;j) = R_driv(t) + 2pi*j*exp(-kt) is exact,
  then hybrid C₀ = ODE C₀. Deviation measures nonlinear coupling.


In [11]:
# ── S4d: Diagnose superposition failure ──
# The decomposition R(t;j) = R_driv(t;0) + 2*pi*j*exp(-kt) fails at ci=61.
# Let's check actual ODE values for different branches at the critical crossings.

ci_targets = [11, 31, 61, 191]
ci_indices = {ci: np.where(coprime_cis == ci)[0][0] for ci in ci_targets}

# Sort branches by key
sorted_br = sorted(res_all.keys())

# Look at R_3 values for a few branches at ci=61
ci_check = 61
idx61 = ci_indices[61]
decay61 = np.exp(-kappa * 61.0)
R3_driv_61 = R_driv_exact[idx61, 3]

print(f"ci = 61, exp(-kappa*61) = {decay61:.6f}")
print(f"R3_driven(61) = {R3_driv_61:.6f}")
print(f"\n{'branch':>20} {'R3_ODE':>12} {'R3_hybrid':>12} {'diff':>10} {'R3_wrap_ODE':>12} {'R3_wrap_hyb':>12}")
print("-" * 82)

# Sample branches: vary j_3 with j_0=j_1=j_2=0
for j3 in range(7):
    br = (0, 0, 0, j3)
    if br in res_all:
        R3_ode = res_all[br][idx61, 3]
        R3_hyb = R3_driv_61 + 2*np.pi * j3 * decay61
        R3w_ode = wrap(R3_ode)
        R3w_hyb = wrap(R3_hyb)
        print(f"{str(br):>20} {R3_ode:12.6f} {R3_hyb:12.6f} {R3_ode-R3_hyb:10.4e} {R3w_ode:12.6f} {R3w_hyb:12.6f}")

# Also check with varied lower indices
print(f"\n--- Vary j_0 with j_3=3 ---")
for j0 in range(2):
    br = (j0, 0, 0, 3)
    if br in res_all:
        R3_ode = res_all[br][idx61, 3]
        R3_hyb = R3_driv_61 + 2*np.pi * 3 * decay61
        R3w_ode = wrap(R3_ode)
        R3w_hyb = wrap(R3_hyb)
        print(f"{str(br):>20} {R3_ode:12.6f} {R3_hyb:12.6f} {R3_ode-R3_hyb:10.4e} {R3w_ode:12.6f} {R3w_hyb:12.6f}")

print(f"\n--- Vary j_2 with j_3=3 ---")
for j2 in range(5):
    br = (0, 0, j2, 3)
    if br in res_all:
        R3_ode = res_all[br][idx61, 3]
        R3_hyb = R3_driv_61 + 2*np.pi * 3 * decay61
        R3w_ode = wrap(R3_ode)
        R3w_hyb = wrap(R3_hyb)
        print(f"{str(br):>20} {R3_ode:12.6f} {R3_hyb:12.6f} {R3_ode-R3_hyb:10.4e} {R3w_ode:12.6f} {R3w_hyb:12.6f}")

ci = 61, exp(-kappa*61) = 0.014855
R3_driven(61) = -0.260948

              branch       R3_ODE    R3_hybrid       diff  R3_wrap_ODE  R3_wrap_hyb
----------------------------------------------------------------------------------
        (0, 0, 0, 0)    -0.260948    -0.260948 0.0000e+00    -0.260948    -0.260948
        (0, 0, 0, 1)    -0.167609    -0.167609 -4.9960e-16    -0.167609    -0.167609
        (0, 0, 0, 2)    -0.074271    -0.074271 5.5511e-16    -0.074271    -0.074271
        (0, 0, 0, 3)     0.019068     0.019068 1.0061e-16     0.019068     0.019068
        (0, 0, 0, 4)     0.112406     0.112406 -6.6613e-16     0.112406     0.112406
        (0, 0, 0, 5)     0.205745     0.205745 -8.8818e-16     0.205745     0.205745
        (0, 0, 0, 6)     0.299083     0.299083 -8.3267e-16     0.299083     0.299083

--- Vary j_0 with j_3=3 ---
        (0, 0, 0, 3)     0.019068     0.019068 1.0061e-16     0.019068     0.019068
        (1, 0, 0, 3)     0.060761     0.019068 4.1693e-02     0.06

In [12]:
# ── S4e: Focused superposition test — all levels at ci=61 ──

ci_check = 61
idx = np.where(coprime_cis == ci_check)[0][0]
decay = np.exp(-kappa * float(ci_check))

# Test: R_k(t;j) vs R_k(t;0) + 2*pi*j_k*exp(-kappa*t)
print(f"ci={ci_check}, exp(-kappa*ci) = {decay:.6f}")
print(f"2*pi*decay = {2*np.pi*decay:.6f}\n")

# Check for branches varying j_3 only (j_0=j_1=j_2=0)
print(f"{'j3':>4} {'R3_ODE':>12} {'R3_driv+trans':>14} {'delta':>12} {'delta/R3':>10}")
print("-" * 56)
R3_driv_val = R_driv_exact[idx, 3]
for j3 in range(7):
    br = (0, 0, 0, j3)
    R3_ode = res_all[br][idx, 3]
    R3_hyb = R3_driv_val + 2*np.pi * j3 * decay
    delta = R3_ode - R3_hyb
    rel = delta/R3_ode if abs(R3_ode) > 1e-10 else 0
    print(f"{j3:4d} {R3_ode:12.6f} {R3_hyb:14.6f} {delta:12.4e} {rel:10.4f}")

# Test with j_2 varied (more coupling)
print(f"\n{'(j0,j1,j2,j3)':>18} {'R3_ODE':>12} {'R3_hyb':>14} {'delta':>12}")
print("-" * 60)
for j2 in range(5):
    br = (0, 0, j2, 3)
    R3_ode = res_all[br][idx, 3]
    R3_hyb = R3_driv_val + 2*np.pi * 3 * decay  # ONLY uses j3=3
    delta = R3_ode - R3_hyb
    print(f"{str(br):>18} {R3_ode:12.6f} {R3_hyb:14.6f} {delta:12.4e}")

print(f"\nKey: delta shows how much R3 depends on lower-level branches (j0,j1,j2)")
print(f"If delta varies with j2, then cross-level coupling is significant.")

ci=61, exp(-kappa*ci) = 0.014855
2*pi*decay = 0.093338

  j3       R3_ODE  R3_driv+trans        delta   delta/R3
--------------------------------------------------------
   0    -0.260948      -0.260948   0.0000e+00    -0.0000
   1    -0.167609      -0.167609  -4.9960e-16     0.0000
   2    -0.074271      -0.074271   5.5511e-16    -0.0000
   3     0.019068       0.019068   1.0061e-16     0.0000
   4     0.112406       0.112406  -6.6613e-16    -0.0000
   5     0.205745       0.205745  -8.8818e-16    -0.0000
   6     0.299083       0.299083  -8.3267e-16    -0.0000

     (j0,j1,j2,j3)       R3_ODE         R3_hyb        delta
------------------------------------------------------------
      (0, 0, 0, 3)     0.019068       0.019068   1.0061e-16
      (0, 0, 1, 3)     0.094981       0.019068   7.5914e-02
      (0, 0, 2, 3)     0.164421       0.019068   1.4535e-01
      (0, 0, 3, 3)     0.235287       0.019068   2.1622e-01
      (0, 0, 4, 3)     0.317993       0.019068   2.9893e-01

Key: del

## S5: Linearized Perturbation System

The decomposition `R(t;j) = R_driv(t) + 2πj₃ exp(−κt)` fails because **lower-level branch indices shift the cascade driving** of R₃. The correct perturbation is:

$$\delta R_k(t;\mathbf{j}) = R_k(t;\mathbf{j}) - R_k^{\text{driv}}(t)$$

with IC $\delta R_k(0) = 2\pi j_k$. Linearizing sin(θ) around the driven solution:

$$\frac{d\delta R_k}{dt} + \kappa \delta R_k = \varepsilon \cos(\theta_k^{\text{driv}}) \cdot \delta\theta_k - \frac{\varepsilon}{p_{k-1}} \cos(\theta_{k-1}^{\text{driv}}) \cdot \delta\theta_{k-1} + \frac{\kappa}{p_{k-1}} \delta R_{k-1}$$

where $\delta\theta_k = \sum_{j=0}^{k-1} \delta R_j \cdot \Pi_j / \Pi_k$. This is a **4×4 linear ODE with time-dependent coefficients** — the Jacobian of the cascade evaluated on the driven trajectory.

In [13]:
# ── S5: Linearized perturbation around driven solution ──

from scipy.integrate import solve_ivp

# First: get driven solution on a DENSE grid for interpolation
t_dense = np.linspace(0, float(P4)+1, 10000)
R0_driv_dense = np.zeros(4)
sol_driv = solve_ivp(
    ssys.cascade_ode, [0, float(P4)+1], R0_driv_dense,
    t_eval=t_dense, method='DOP853', rtol=1e-12, atol=1e-14
)
R_driv_interp = sol_driv.y  # (4, 10000)

# Interpolation function for driven R and theta
from scipy.interpolate import interp1d
R_driv_fn = interp1d(t_dense, R_driv_interp, axis=1, kind='cubic', fill_value='extrapolate')

def theta_from_R(R_vals, t):
    """Reconstruct theta from R values and time."""
    th = np.zeros(5)
    th[0] = omega * t
    for k in range(4):
        th[k+1] = (R_vals[k] + th[k]) / primes[k]
    return th

def linearized_ode(t, dR):
    """
    dδR/dt = Jacobian · δR (evaluated on driven trajectory)
    
    The Jacobian captures how lower-level perturbations propagate
    through sin(theta) coupling into higher levels.
    """
    R_d = R_driv_fn(t)
    th_d = theta_from_R(R_d, t)
    
    # delta_theta[k] = sum_{j=0}^{k-1} dR[j] * Pi[j] / Pi[k]
    dtheta = np.zeros(5)  # 5 levels (including theta_0 which has dtheta=0)
    for k in range(1, 5):
        for j in range(k):
            dtheta[k] += dR[j] * Pi[j] / Pi[k]
    
    ddR = np.zeros(4)
    # Level 0: d(dR0)/dt = eps*cos(th0)*dtheta[0] - kappa*dR[0]
    # But dtheta[0] = 0 (theta_0 = omega*t, no perturbation)
    ddR[0] = -kappa * dR[0]
    
    # Level k >= 1
    for k in range(1, 4):
        # eps*cos(th_k)*dtheta[k] - (eps/p_{k-1})*cos(th_{k-1})*dtheta[k-1]
        # + (kappa/p_{k-1})*dR[k-1] - kappa*dR[k]
        ddR[k] = (
            epsilon * np.cos(th_d[k]) * dtheta[k]
            - epsilon * np.cos(th_d[k-1]) * dtheta[k-1] / primes[k-1]
            + kappa * dR[k-1] / primes[k-1]
            - kappa * dR[k]
        )
    
    return ddR

# Integrate linearized system for each branch
t_eval_ci = coprime_cis.astype(float)
dR_per_branch = {}  # branch -> (N_ci, 4) array of delta R

t0 = time.time()
for br in all_branches:
    dR0 = np.array([2*np.pi * j_k for j_k in br], dtype=float)
    sol = solve_ivp(
        linearized_ode, [0, float(P4)+1], dR0,
        t_eval=t_eval_ci, method='DOP853', rtol=1e-10, atol=1e-12
    )
    dR_per_branch[br] = sol.y.T  # (N_ci, 4)
dt_lin = time.time() - t0
print(f"Linearized system: 210 branches in {dt_lin:.1f}s")

# Reconstruct full R: R(t;j) = R_driv(t) + delta_R(t;j)
# Compute branch-averaged R² at each crossing
R_sq_lin = np.zeros((N_ci, 4))
for br in all_branches:
    dR = dR_per_branch[br]  # (N_ci, 4)
    R_full = R_driv_exact + dR  # (N_ci, 4)
    R_w = np.mod(R_full + np.pi, 2*np.pi) - np.pi
    R_sq_lin += R_w**2
R_sq_lin /= len(all_branches)

# Sector accumulation
cum_sq_lin = np.zeros((48, N_ci, 4))
cum_ct_lin = np.zeros((48, N_ci), dtype=np.int64)
for j in range(N_ci):
    if j > 0:
        cum_sq_lin[:, j, :] = cum_sq_lin[:, j-1, :]
        cum_ct_lin[:, j] = cum_ct_lin[:, j-1]
    s = sector_idx[j]
    cum_sq_lin[s, j, :] += R_sq_lin[j, :]
    cum_ct_lin[s, j] += 1

# C₀ at R4
j_end = N_ci - 1
lev = 3

g1q = cum_sq_lin[QUARK_G1, j_end, lev] / max(cum_ct_lin[QUARK_G1, j_end], 1)
g2q = cum_sq_lin[QUARK_G2, j_end, lev] / max(cum_ct_lin[QUARK_G2, j_end], 1)
C0_Q_lin = np.sqrt(g1q / g2q)

g1l = cum_sq_lin[LEPTON_G1, j_end, lev] / max(cum_ct_lin[LEPTON_G1, j_end], 1)
g2l = cum_sq_lin[LEPTON_G2, j_end, lev] / max(cum_ct_lin[LEPTON_G2, j_end], 1)
C0_L_lin = np.sqrt(g1l / g2l)

amp_Q_lin = C0_Q_lin / cp_trans_q
amp_L_lin = C0_L_lin / cp_trans_l

print(f"\n{'':>14} {'C0_lin':>12} {'C0_ODE':>12} {'err%':>10} {'amp':>10}")
print("-" * 60)
print(f"{'QUARK':>14} {C0_Q_lin:12.6f} {ODE_C0_Q:12.6f} {(C0_Q_lin/ODE_C0_Q-1)*100:9.4f}% {amp_Q_lin:10.6f}")
print(f"{'LEPTON':>14} {C0_L_lin:12.6f} {ODE_C0_L:12.6f} {(C0_L_lin/ODE_C0_L-1)*100:9.4f}% {amp_L_lin:10.6f}")

# Verify linearized delta_R against actual (ODE - driven) for a test branch
test_br = (0, 0, 4, 3)
dR_lin = dR_per_branch[test_br]
dR_exact = res_all[test_br][:N_ci] - R_driv_exact
idx61 = np.where(coprime_cis == 61)[0][0]
print(f"\nValidation: branch {test_br} at ci=61:")
for lev_i in range(4):
    print(f"  R{lev_i}: dR_lin={dR_lin[idx61,lev_i]:.6f}, dR_exact={dR_exact[idx61,lev_i]:.6f}, "
          f"err={dR_lin[idx61,lev_i]-dR_exact[idx61,lev_i]:.4e}")

Linearized system: 210 branches in 26.4s

                     C0_lin       C0_ODE       err%        amp
------------------------------------------------------------
         QUARK     6.338978     6.606742   -4.0529%   2.958634
        LEPTON     5.797035     5.911955   -1.9439%   4.768753

Validation: branch (0, 0, 4, 3) at ci=61:
  R0: dR_lin=0.000000, dR_exact=0.000000, err=-6.6650e-14
  R1: dR_lin=0.000000, dR_exact=-0.000000, err=3.6051e-14
  R2: dR_lin=0.373354, dR_exact=0.373354, err=-2.2204e-16
  R3: dR_lin=0.599164, dR_exact=0.578941, err=2.0223e-02


In [14]:
# ── S5b: State-transition matrix and analytic branch average ──
# In the linearized system: dR(t;j) = Phi(t,0) * dR(0;j)
# where dR(0;j) = (2*pi*j0, 2*pi*j1, 2*pi*j2, 2*pi*j3)
# 
# Extract Phi by integrating with unit initial conditions.

# Compute state-transition matrix columns
Phi_cols = {}  # ci_idx -> Phi matrix (4x4)

# Integrate with e_k initial conditions
unit_results = {}
for k in range(4):
    dR0 = np.zeros(4)
    dR0[k] = 1.0  # unit perturbation in level k
    sol_k = solve_ivp(
        linearized_ode, [0, float(P4)+1], dR0,
        t_eval=t_eval_ci, method='DOP853', rtol=1e-10, atol=1e-12
    )
    unit_results[k] = sol_k.y.T  # (N_ci, 4)

# Build Phi(ci, 0) for each crossing
Phi_at_ci = np.zeros((N_ci, 4, 4))  # Phi[ci_idx, row, col]
for k in range(4):
    Phi_at_ci[:, :, k] = unit_results[k]  # column k of Phi

# The linearized dR at each crossing for branch j is:
# dR(ci;j) = Phi(ci) @ [2*pi*j0, 2*pi*j1, 2*pi*j2, 2*pi*j3]
# So R_3(ci;j) = R_driv(ci,3) + 2*pi * sum_k Phi(ci, 3, k) * j_k

# Branch-averaged R_3^2 WITHOUT wrapping:
# <R_3^2> = (R_d + 2*pi * sum phi_{3k} <j_k>)^2 + (2*pi)^2 * sum_{k,l} phi_{3k} phi_{3l} Cov(j_k, j_l)
# j_k independent:  <j_k> = (p_k-1)/2,  Var(j_k) = (p_k^2-1)/12

j_mean = np.array([(p-1)/2 for p in primes])  # [0.5, 1, 2, 3]
j_var = np.array([(p**2 - 1)/12 for p in primes])  # [0.25, 0.667, 2, 4]

R_sq_analytic = np.zeros((N_ci, 4))
for ci_idx in range(N_ci):
    for lev in range(4):
        phi_row = Phi_at_ci[ci_idx, lev, :]  # (4,) — row lev of Phi
        R_d = R_driv_exact[ci_idx, lev]
        mean_dR = 2*np.pi * np.sum(phi_row * j_mean)
        mean_R = R_d + mean_dR
        var_dR = (2*np.pi)**2 * np.sum(phi_row**2 * j_var)
        R_sq_analytic[ci_idx, lev] = mean_R**2 + var_dR

# Compare to ODE branch average (without wrapping first)
R_sq_nowrap = np.zeros((N_ci, 4))
for br in all_branches:
    R_full = res_all[br][:N_ci]  # (N_ci, 4) — RAW, no wrapping
    R_sq_nowrap += R_full**2
R_sq_nowrap /= len(all_branches)

print("=" * 70)
print("S5b: STATE-TRANSITION MATRIX — ANALYTIC <R²> vs ODE (no wrapping)")
print("=" * 70)
print(f"\n{'ci':>6} {'R²_analytic':>12} {'R²_ODE_nw':>12} {'R²_ODE_wrap':>12} {'err_nw%':>10} {'err_wrap%':>10}")
print("-" * 65)
lev = 3
for ci_val in [1, 11, 31, 61, 191, 209]:
    idx = np.where(coprime_cis == ci_val)[0][0]
    ra = R_sq_analytic[idx, lev]
    rn = R_sq_nowrap[idx, lev]
    rw = R_sq_avg[idx, lev]
    en = (ra/rn - 1)*100 if rn > 0 else 0
    ew = (ra/rw - 1)*100 if rw > 0 else 0
    print(f"{ci_val:6d} {ra:12.6e} {rn:12.6e} {rw:12.6e} {en:9.3f}% {ew:9.3f}%")

# The state-transition matrix at R3 level:
print(f"\nState-transition matrix row 3 (R3 <- dR_k) at key crossings:")
print(f"{'ci':>6} {'Phi(3,0)':>10} {'Phi(3,1)':>10} {'Phi(3,2)':>10} {'Phi(3,3)':>10}")
print("-" * 50)
for ci_val in [11, 31, 61, 191]:
    idx = np.where(coprime_cis == ci_val)[0][0]
    phi = Phi_at_ci[idx, 3, :]
    print(f"{ci_val:6d} {phi[0]:10.4e} {phi[1]:10.4e} {phi[2]:10.4e} {phi[3]:10.4e}")

# How important is each level's contribution?
print(f"\n  Contribution to R3 variance by source level:")
for ci_val in [11, 61]:
    idx = np.where(coprime_cis == ci_val)[0][0]
    phi = Phi_at_ci[idx, 3, :]
    contribs = (2*np.pi)**2 * phi**2 * j_var
    total = contribs.sum()
    print(f"  ci={ci_val}: ", end="")
    for k in range(4):
        print(f"R{k}={contribs[k]/total*100:.1f}% ", end="")
    print()

S5b: STATE-TRANSITION MATRIX — ANALYTIC <R²> vs ODE (no wrapping)

    ci  R²_analytic    R²_ODE_nw  R²_ODE_wrap    err_nw%  err_wrap%
-----------------------------------------------------------------
     1 4.589332e+02 4.528281e+02 1.905024e+00     1.348% 23990.686%
    11 1.466598e+02 1.286915e+02 3.409538e+00    13.962%  4201.456%
    31 1.018708e+01 1.007068e+01 3.895099e+00     1.156%   161.536%
    61 1.160438e-01 1.114439e-01 1.114439e-01     4.128%     4.128%
   191 7.811354e-02 7.811256e-02 7.811256e-02     0.001%     0.001%
   209 9.140685e-02 9.140717e-02 9.140717e-02    -0.000%    -0.000%

State-transition matrix row 3 (R3 <- dR_k) at key crossings:
    ci   Phi(3,0)   Phi(3,1)   Phi(3,2)   Phi(3,3)
--------------------------------------------------
    11 6.3687e-03 1.8571e-02 9.3451e-02 4.6810e-01
    31 7.9671e-03 1.9196e-02 5.1913e-02 1.1775e-01
    61 6.6874e-03 9.0681e-03 1.2699e-02 1.4855e-02
   191 2.5536e-05 1.1350e-05 5.0662e-06 1.8875e-06

  Contribution to R3 v

In [15]:
# ── S5c: Full linearized C₀ with wrapping ──
# Use: R_3(ci;j) = R_driv(ci) + 2*pi * sum_k Phi(ci,3,k) * j_k
# Then wrap and average over all 210 branches.

R_sq_lin_wrap = np.zeros((N_ci, 4))
for ci_idx in range(N_ci):
    for lev in range(4):
        R_d = R_driv_exact[ci_idx, lev]
        phi_row = Phi_at_ci[ci_idx, lev, :]
        sq_sum = 0.0
        count = 0
        # Iterate over ALL branches
        for br in all_branches:
            j_arr = np.array(br, dtype=float)
            dR = 2*np.pi * np.dot(phi_row, j_arr)
            R_total = R_d + dR
            R_w = wrap(R_total)
            sq_sum += R_w**2
            count += 1
        R_sq_lin_wrap[ci_idx, lev] = sq_sum / count

# Sector C₀
cum_sq_lw = np.zeros((48, N_ci, 4))
cum_ct_lw = np.zeros((48, N_ci), dtype=np.int64)
for j in range(N_ci):
    if j > 0:
        cum_sq_lw[:, j, :] = cum_sq_lw[:, j-1, :]
        cum_ct_lw[:, j] = cum_ct_lw[:, j-1]
    s = sector_idx[j]
    cum_sq_lw[s, j, :] += R_sq_lin_wrap[j, :]
    cum_ct_lw[s, j] += 1

j_end = N_ci - 1
lev = 3

g1q = cum_sq_lw[QUARK_G1, j_end, lev] / max(cum_ct_lw[QUARK_G1, j_end], 1)
g2q = cum_sq_lw[QUARK_G2, j_end, lev] / max(cum_ct_lw[QUARK_G2, j_end], 1)
C0_Q_lw = np.sqrt(g1q / g2q)

g1l = cum_sq_lw[LEPTON_G1, j_end, lev] / max(cum_ct_lw[LEPTON_G1, j_end], 1)
g2l = cum_sq_lw[LEPTON_G2, j_end, lev] / max(cum_ct_lw[LEPTON_G2, j_end], 1)
C0_L_lw = np.sqrt(g1l / g2l)

amp_Q_lw = C0_Q_lw / cp_trans_q
amp_L_lw = C0_L_lw / cp_trans_l

print("=" * 70)
print("S5c: LINEARIZED C₀ WITH WRAPPING (state-transition matrix + wrap)")
print("=" * 70)

print(f"\n{'Method':>18} {'C0_Q':>10} {'errQ%':>8} {'ampQ':>10} {'C0_L':>10} {'errL%':>8} {'ampL':>10}")
print("-" * 76)
methods = [
    ('zeroth-order', C0_Q_z, C0_L_z, amp_Q_z, amp_L_z),
    ('1st-order pert', C0_Q_p, C0_L_p, amp_Q_p, amp_L_p),
    ('hybrid (driv+exp)', C0_Q_h, C0_L_h, amp_Q_h, amp_L_h),
    ('linearized (S5)', C0_Q_lin, C0_L_lin, amp_Q_lin, amp_L_lin),
    ('lin+wrap (S5c)', C0_Q_lw, C0_L_lw, amp_Q_lw, amp_L_lw),
    ('ODE (truth)', ODE_C0_Q, ODE_C0_L, 3.0836096405, 4.8632881895),
]
for name, cq, cl, aq, al in methods:
    eq = (cq/ODE_C0_Q - 1)*100
    el = (cl/ODE_C0_L - 1)*100
    print(f"{name:>18} {cq:10.4f} {eq:7.3f}% {aq:10.4f} {cl:10.4f} {el:7.3f}% {al:10.4f}")

print(f"\nGram targets:     ampQ = {TARGET_AMP_Q:.4f}, ampL = {TARGET_AMP_L:.4f}")

# Per-crossing comparison
print(f"\n{'ci':>6} {'R²_lw':>12} {'R²_ODE':>12} {'err%':>10}")
print("-" * 42)
for ci_val in [1, 11, 31, 61, 191]:
    idx = np.where(coprime_cis == ci_val)[0][0]
    rl = R_sq_lin_wrap[idx, lev]
    ro = R_sq_avg[idx, lev]
    d = (rl/ro - 1)*100 if ro > 0 else 0
    print(f"{ci_val:6d} {rl:12.6e} {ro:12.6e} {d:9.4f}%")

S5c: LINEARIZED C₀ WITH WRAPPING (state-transition matrix + wrap)

            Method       C0_Q    errQ%       ampQ       C0_L    errL%       ampL
----------------------------------------------------------------------------
      zeroth-order 41393.6593 626436.615% 19319.9435     5.7814  -2.209%     4.7559
    1st-order pert     6.7407   2.028%     3.1461    10.1678  71.987%     8.3642
 hybrid (driv+exp)     6.7372   1.974%     3.1445    10.1723  72.063%     8.3679
   linearized (S5)     6.3390  -4.053%     2.9586     5.7970  -1.944%     4.7688
    lin+wrap (S5c)     6.3390  -4.053%     2.9586     5.7970  -1.944%     4.7688
       ODE (truth)     6.6067   0.000%     3.0836     5.9120   0.000%     4.8633

Gram targets:     ampQ = 3.0764, ampL = 4.8990

    ci        R²_lw       R²_ODE       err%
------------------------------------------
     1 1.607981e+00 1.905024e+00  -15.5926%
    11 3.138808e+00 3.409538e+00   -7.9404%
    31 3.899722e+00 3.895099e+00    0.1187%
    61 1.160438e-0

## S6: Analytic State-Transition Matrix

At first order in ε, cos(θ_k^driv) ≈ cos(ωt/Πₖ). The linearized perturbation system becomes:

$$\frac{d\delta R_k}{dt} + \kappa \cdot \delta R_k = \text{coupling from } \delta R_0, \ldots, \delta R_{k-1}$$

Since δR₀ = 2πj₀ exp(−κt) exactly, and the coupling introduces terms exp(−κt)·cos(Ωt) and exp(−κt)·const, the general solution has the form:

$$\delta R_k(t) = e^{-\kappa t} \cdot \sum_m \left[2\pi j_m \cdot \Phi_{k,m}^{\text{(an)}}(t)\right]$$

where Φ^(an)_{k,m}(t) are analytically computable functions involving polynomials × sin and cos terms. The key is the **resonant forcing**: exp(−κt) driving into a system with damping rate κ creates **secular growth** (linear in t) before exponential decay.

In [16]:
# ── S6: Analytic state-transition matrix ──
# At first order: delta_theta_k depends on sum of lower-level dR.
# theta_k = wt/Pi_k + (small corrections from R). At first order cos(theta_k) ~ cos(wt/Pi_k).
#
# Level 0: d(dR0)/dt = -kappa*dR0  =>  dR0 = 2*pi*j0*exp(-kt)
#
# Level 1: The cascade_ode level-1 RHS involves:
#   eps*sin(theta_1) - eps*sin(theta_0)/p0 + kappa*R0/p0 - kappa*R1
# Linearizing around the driven solution:
#   eps*cos(theta_1^d)*delta_theta_1 - eps*cos(theta_0^d)*delta_theta_0/p0
#   + kappa*dR0/p0 - kappa*dR1
# where delta_theta_0 = 0 (theta_0 = wt, no perturbation)
# and delta_theta_1 = dR0 / (Pi_1) = dR0 / 2  (since Pi_1 = p0 = 2)
#
# So: d(dR1)/dt + kappa*dR1 = eps*cos(wt/2)*dR0/2 + kappa*dR0/2
# = dR0 * [eps*cos(wt/2)/2 + kappa/2]
# = 2*pi*j0*exp(-kt) * [rho*cos(pi*t)/2 + rho/2]  since eps=kappa=rho, w/2=pi
#
# This is dR1/dt + kappa*dR1 = A*exp(-kt)*cos(Omega*t) + B*exp(-kt)
# where A = pi*j0*rho, B = pi*j0*rho, Omega = pi.
#
# For dR/dt + kR = exp(-kt) * g(t):
# Let dR = exp(-kt)*u(t) => u'(t) = g(t)  (resonant forcing!)
# u(t) = integral g(t) dt + const
#
# For g(t) = A*cos(Omega*t) + B:
# u(t) = A/Omega * sin(Omega*t) + B*t + C

def analytic_Phi(t_arr):
    """
    Compute analytic state-transition matrix elements at first order.
    Returns Phi_an[ci_idx, row, col] where
    dR_row(t) = exp(-kappa*t) * sum_col 2*pi * Phi_an(t, row, col) * j_col
    
    Note: Phi_an incorporates the exp(-kt) already, so
    actual Phi(t, row, col) = exp(-kappa*t) * Phi_an_internal(t, row, col)
    But we return the FULL Phi (with exp(-kt) included).
    """
    N = len(t_arr)
    Phi = np.zeros((N, 4, 4))
    rho = kappa  # kappa = epsilon = rho
    
    # Level 0: dR0 = j0 * exp(-kt)  =>  Phi(0,0) = exp(-kt)
    Phi[:, 0, 0] = np.exp(-kappa * t_arr)
    
    # Level 1: 
    # dR1 from j1: direct transient, dR1/dt + kR1 = 0 with IC=2pi*j1
    # => Phi(1,1) = exp(-kt)
    Phi[:, 1, 1] = np.exp(-kappa * t_arr)
    
    # dR1 from j0: resonant forcing
    # u1_from_j0(t) = rho/(2*pi) * sin(pi*t) + rho/2 * t
    # dR1 = exp(-kt) * u1_from_j0(t) * 2*pi*j0
    # Phi(1,0) = exp(-kt) * [rho/(2*pi)*sin(pi*t) + rho/2*t]
    # Wait — need to be more careful about the coefficient.
    #
    # d(dR1)/dt + k*dR1 = alpha * dR0
    # where alpha(t) = rho*cos(pi*t)/2 + rho/2
    # dR0 = 2*pi*j0*exp(-kt)
    # So the forcing is: 2*pi*j0*exp(-kt) * alpha(t)
    # Let dR1 = exp(-kt)*u(t): u'(t) = 2*pi*j0 * alpha(t) + IC correction
    # Actually for the UNIT response (j0=1, j1=0):
    # IC: dR1(0) = 0 => u(0) = 0
    # u(t) = 2*pi * integral_0^t alpha(s) ds
    # = 2*pi * integral_0^t [rho*cos(pi*s)/2 + rho/2] ds
    # = 2*pi * [rho*sin(pi*t)/(2*pi) + rho*t/2]
    # = rho*sin(pi*t)/pi + pi*rho*t
    
    # Hmm, wait. Let me redo more carefully.
    # For the unit j0=1 initial condition: dR0 = 2*pi * exp(-kt)
    # Forcing of dR1: F(t) = alpha(t) * dR0(t) = alpha(t) * 2*pi * exp(-kt)
    # where alpha(t) = rho*cos(pi*t)/2 + rho/2
    # dR1/dt + k*dR1 = F(t)
    # Let dR1 = exp(-kt)*u(t): u'(t) = 2*pi*alpha(t) = pi*rho*cos(pi*t) + pi*rho
    # u(t) = rho*sin(pi*t) + pi*rho*t  (since u(0) = dR1_0/exp(0) = 0)
    
    u10 = rho * np.sin(np.pi * t_arr) + np.pi * rho * t_arr
    Phi[:, 1, 0] = np.exp(-kappa * t_arr) * u10 / (2*np.pi)
    # The /2pi normalizes because Phi maps j (not 2*pi*j)
    # Actually, Phi should map: dR_row(t) = sum_col Phi(row,col) * 2*pi*j_col
    # For j0=1, dR1(t) = exp(-kt) * u10(t) where u10 was computed with pre-factor 2*pi
    # So Phi(1,0) * 2*pi = exp(-kt) * u10 => Phi(1,0) = exp(-kt) * u10 / (2*pi)
    
    # Actually let me reorganize. I want Phi such that:
    # dR(t) = Phi(t) @ (2*pi*j)
    # So f_k(t;j) = sum_m Phi(t,k,m) * 2*pi*j_m
    # For unit e_0 basis: f_k(t; j=(1,0,0,0)) = Phi(t,k,0) * 2*pi
    # I need: f_1(t; j0=1) = dR1(t when IC dR=(2pi,0,0,0))
    #   = exp(-kt) * u10(t)
    # => Phi(1,0) * 2*pi = exp(-kt)*u10(t)
    # => Phi(1,0) = exp(-kt)*u10/(2*pi)
    
    # Ok, this is correct. But u10(t) was computed as:
    # u'(t) = 2*pi * alpha(t), so u10(t) = 2*pi * integral alpha
    # = 2*pi * [rho*sin(pi*t)/(2*pi) + rho*t/2] = rho*sin(pi*t) + pi*rho*t
    # So Phi(1,0) = exp(-kt) * (rho*sin(pi*t) + pi*rho*t) / (2*pi)
    
    # Level 2: similar cascade
    # d(dR2)/dt + k*dR2 = eps*cos(wt/6)*delta_theta_2 - eps*cos(wt/2)*delta_theta_1/3 + k*dR1/3
    # delta_theta_1 = dR0/2, delta_theta_2 = dR0/6 + dR1/6
    # So: forcing = eps*cos(wt/6)*(dR0/6 + dR1/6) - eps*cos(wt/2)*dR0/(2*3) + k*dR1/3
    # = eps*cos(wt/6)*dR0/6 + eps*cos(wt/6)*dR1/6 - eps*cos(wt/2)*dR0/6 + k*dR1/3
    
    # For unit j2=1 (IC dR=(0,0,2pi,0)):
    # Only dR2 is nonzero at IC, with no forcing from dR0,dR1 (both 0)
    # => Phi(2,2) = exp(-kt)
    Phi[:, 2, 2] = np.exp(-kappa * t_arr)
    
    # For unit j3=1: Phi(3,3) = exp(-kt) similarly
    Phi[:, 3, 3] = np.exp(-kappa * t_arr)
    
    # Cross terms: Phi(2,0), Phi(2,1), Phi(3,0), Phi(3,1), Phi(3,2)
    # These involve cascaded resonant forcing — omitting for now and using
    # the NUMERICAL Phi to validate the diagonal and (1,0) terms.
    
    # For off-diagonal: use numerical Phi from S5b
    # (we'll compute the analytic versions in the next step)
    
    return Phi

Phi_an = analytic_Phi(t_eval_ci)

# Compare analytic vs numerical Phi at key crossings
print("=" * 70)
print("S6: ANALYTIC vs NUMERICAL STATE-TRANSITION MATRIX (row 3)")
print("=" * 70)
print(f"\n{'ci':>6} {'comp':>6} {'Phi_num':>12} {'Phi_ana':>12} {'err%':>10}")
print("-" * 50)
for ci_val in [11, 31, 61, 191]:
    idx = np.where(coprime_cis == ci_val)[0][0]
    for col in range(4):
        pn = Phi_at_ci[idx, 3, col]
        pa = Phi_an[idx, 3, col]
        err = (pa/pn - 1)*100 if abs(pn) > 1e-15 else 0
        lbl = f"Phi(3,{col})"
        print(f"{ci_val:6d} {lbl:>6} {pn:12.4e} {pa:12.4e} {err:9.2f}%")
    print()

# Also check Phi(1,0) — the first cross-level coupling
print("Phi(1,0) — first cross-level coupling:")
for ci_val in [11, 31, 61]:
    idx = np.where(coprime_cis == ci_val)[0][0]
    pn = Phi_at_ci[idx, 1, 0]
    pa = Phi_an[idx, 1, 0]
    err = (pa/pn - 1)*100 if abs(pn) > 1e-15 else 0
    print(f"  ci={ci_val}: num={pn:.6e}, ana={pa:.6e}, err={err:.2f}%")

S6: ANALYTIC vs NUMERICAL STATE-TRANSITION MATRIX (row 3)

    ci   comp      Phi_num      Phi_ana       err%
--------------------------------------------------
    11 Phi(3,0)   6.3687e-03   0.0000e+00   -100.00%
    11 Phi(3,1)   1.8571e-02   0.0000e+00   -100.00%
    11 Phi(3,2)   9.3451e-02   0.0000e+00   -100.00%
    11 Phi(3,3)   4.6810e-01   4.6810e-01     -0.00%

    31 Phi(3,0)   7.9671e-03   0.0000e+00   -100.00%
    31 Phi(3,1)   1.9196e-02   0.0000e+00   -100.00%
    31 Phi(3,2)   5.1913e-02   0.0000e+00   -100.00%
    31 Phi(3,3)   1.1775e-01   1.1775e-01     -0.00%

    61 Phi(3,0)   6.6874e-03   0.0000e+00   -100.00%
    61 Phi(3,1)   9.0681e-03   0.0000e+00   -100.00%
    61 Phi(3,2)   1.2699e-02   0.0000e+00   -100.00%
    61 Phi(3,3)   1.4855e-02   1.4855e-02     -0.00%

   191 Phi(3,0)   2.5536e-05   0.0000e+00   -100.00%
   191 Phi(3,1)   1.1350e-05   0.0000e+00   -100.00%
   191 Phi(3,2)   5.0662e-06   0.0000e+00   -100.00%
   191 Phi(3,3)   1.8875e-06   1.8875e-06

In [17]:
# ── S6b: Compact Phi comparison ──
# Compare the diagonal and (1,0) elements only (the parts we computed analytically)

print("Diagonal elements Phi(k,k) — should be exp(-kappa*ci):")
for ci_val in [11, 61, 191]:
    idx = np.where(coprime_cis == ci_val)[0][0]
    exp_val = np.exp(-kappa * float(ci_val))
    for k in range(4):
        num = Phi_at_ci[idx, k, k]
        err = (num/exp_val - 1)*100
        # Only print if error is interesting
    print(f"  ci={ci_val}: exp(-kt) = {exp_val:.6e}, Phi diag err: "
          f"lev0={((Phi_at_ci[idx,0,0]/exp_val)-1)*100:.2f}%, "
          f"lev3={((Phi_at_ci[idx,3,3]/exp_val)-1)*100:.2f}%")

print(f"\nCross-coupling Phi(1,0) — analytic: exp(-kt)*[rho*sin(pi*t)+pi*rho*t]/(2pi)")
for ci_val in [11, 31, 61]:
    idx = np.where(coprime_cis == ci_val)[0][0]
    pn = Phi_at_ci[idx, 1, 0]
    pa = Phi_an[idx, 1, 0]
    print(f"  ci={ci_val}: num={pn:.6e}, ana={pa:.6e}, err={(pa/pn-1)*100:.2f}%")

# KEY: the Phi(3,2) element (R2->R3 coupling) at ci_val=61
# This is the dominant cross-level term for LEPTON
print(f"\nPhi(3,2) at physical crossings (R2->R3, dominant for LEPTON):")
for ci_val in [11, 31, 61]:
    idx = np.where(coprime_cis == ci_val)[0][0]
    pn = Phi_at_ci[idx, 3, 2]
    print(f"  ci={ci_val}: Phi(3,2) = {pn:.6e}")

# Relative importance: which Phi(3,k) sets the C₀?
idx11 = np.where(coprime_cis == 11)[0][0]  # Q_g1
idx191 = np.where(coprime_cis == 191)[0][0]  # Q_g2
idx31 = np.where(coprime_cis == 31)[0][0]  # L_g1
idx61 = np.where(coprime_cis == 61)[0][0]  # L_g2

print(f"\nPhi(3,:) at physical crossings:")
print(f"  ci=11  (Q_g1): {Phi_at_ci[idx11, 3, :]}")
print(f"  ci=191 (Q_g2): {Phi_at_ci[idx191, 3, :]}")
print(f"  ci=31  (L_g1): {Phi_at_ci[idx31, 3, :]}")
print(f"  ci=61  (L_g2): {Phi_at_ci[idx61, 3, :]}")

Diagonal elements Phi(k,k) — should be exp(-kappa*ci):
  ci=11: exp(-kt) = 4.681006e-01, Phi diag err: lev0=0.00%, lev3=0.00%
  ci=61: exp(-kt) = 1.485528e-02, Phi diag err: lev0=-0.00%, lev3=0.00%
  ci=191: exp(-kt) = 1.887510e-06, Phi diag err: lev0=-0.00%, lev3=0.00%

Cross-coupling Phi(1,0) — analytic: exp(-kt)*[rho*sin(pi*t)+pi*rho*t]/(2pi)
  ci=11: num=1.776008e-01, ana=1.776610e-01, err=0.03%
  ci=31: num=1.259315e-01, ana=1.259441e-01, err=0.01%
  ci=61: num=3.126440e-02, ana=3.126591e-02, err=0.00%

Phi(3,2) at physical crossings (R2->R3, dominant for LEPTON):
  ci=11: Phi(3,2) = 9.345054e-02
  ci=31: Phi(3,2) = 5.191337e-02
  ci=61: Phi(3,2) = 1.269853e-02

Phi(3,:) at physical crossings:
  ci=11  (Q_g1): [0.00636867 0.01857124 0.09345054 0.46810057]
  ci=191 (Q_g2): [2.55361332e-05 1.13496829e-05 5.06618871e-06 1.88750982e-06]
  ci=31  (L_g1): [0.00796708 0.01919557 0.05191337 0.11774862]
  ci=61  (L_g2): [0.00668738 0.00906813 0.01269853 0.01485528]


In [18]:
# ── S6c: Complete analytic state-transition matrix ──
# All elements computed via cascaded resonant forcing.
#
# General pattern: for Phi(k,m) with m < k and no intermediate nonzero ICs,
# the forcing of level k from level m's transient goes through:
#   Level m: dR_m = 2*pi*j_m * exp(-kappa*t) (direct decay)
#   The forcing chain m -> m+1 -> ... -> k propagates through delta_theta.
#
# Key formula: for dR/dt + kappa*R = 2*pi*exp(-kt)*g(t) with R(0)=0,
# R(t) = exp(-kt) * integral_0^t g(s) ds    [resonant forcing]

def analytic_Phi_full(t_arr):
    """Complete 4x4 analytic state-transition matrix at first order."""
    N = len(t_arr)
    Phi = np.zeros((N, 4, 4))
    t = t_arr
    rho = kappa
    
    # Diagonal: Phi(k,k) = exp(-kappa*t)
    for k in range(4):
        Phi[:, k, k] = np.exp(-kappa * t)
    
    # ── Level 1 cross-terms ──
    # Phi(1,0): from S6 derivation
    # u10(t) = rho*sin(pi*t) + pi*rho*t  (integral of alpha)
    # Phi(1,0) = exp(-kt) * u10 / (2*pi)
    u10 = rho * np.sin(np.pi * t) + np.pi * rho * t
    Phi[:, 1, 0] = np.exp(-kappa * t) * u10 / (2*np.pi)
    
    # ── Level 2 cross-terms (Phi(2,0), Phi(2,1)) ──
    # Forcing of level 2: eps*cos(wt/6)*dtheta_2 - eps*cos(wt/2)*dtheta_1/3 + kappa*dR1/3
    #
    # For unit j1 (IC: dR=(0, 2pi, 0, 0)):
    #   dR0=0, dR1=2*pi*exp(-kt), dR2=0
    #   dtheta_1 = dR0/Pi_1 = 0
    #   dtheta_2 = dR0*Pi_0/Pi_2 + dR1*Pi_1/Pi_2 = 0 + dR1*2/6 = dR1/3
    #   Forcing = eps*cos(wt/6)*dR1/3 - 0 + kappa*dR1/3
    #   = dR1 * [eps*cos(wt/6)/3 + kappa/3]
    #   = 2*pi*exp(-kt) * rho/3 * [cos(wt/6) + 1]
    # Resonant: u21(t) = integral 2*pi*rho/3 * [cos(2*pi*t/6) + 1] dt
    #   = 2*pi*rho/3 * [6*sin(2*pi*t/6)/(2*pi) + t]
    #   = 2*pi*rho/3 * [3*sin(pi*t/3)/pi + t]
    # Phi(2,1) = exp(-kt) * u21 / (2*pi)
    Omega_6 = omega / 6  # 2*pi/6 = pi/3
    u21 = 2*np.pi*rho/3 * (np.sin(Omega_6 * t) / Omega_6 + t)
    Phi[:, 2, 1] = np.exp(-kappa * t) * u21 / (2*np.pi)
    
    # For unit j0 (IC: dR=(2pi, 0, 0, 0)):
    #   dR0=2*pi*exp(-kt), dR1 from Phi(1,0), dR2=0
    #   dtheta_1 = dR0/2
    #   dtheta_2 = dR0/6 + dR1*2/6 = dR0/6 + dR1/3
    #   Forcing = eps*cos(wt/6)*(dR0/6 + dR1/3) - eps*cos(wt/2)*dR0/(2*3) + kappa*dR1/3
    # This involves dR1(t) which includes the cross-coupling u10 term.
    # At first order (dropping O(rho^2) coupling terms):
    #   dR1 ≈ 0 (it's O(rho) via coupling), dtheta_2 ≈ dR0/6
    #   Forcing ≈ eps*cos(wt/6)*dR0/6 - eps*cos(wt/2)*dR0/6 + 0
    #   = 2*pi*exp(-kt) * rho/6 * [cos(wt/6) - cos(wt/2)]
    # But the kappa*dR1/3 term is also O(rho^2), so drop it.
    #   u20(t) = 2*pi*rho/6 * integral [cos(2*pi*t/6) - cos(pi*t)] dt
    #   = 2*pi*rho/6 * [sin(2*pi*t/6)/(2*pi/6) - sin(pi*t)/pi]
    #   = 2*pi*rho/6 * [3*sin(pi*t/3)/pi - sin(pi*t)/pi]
    u20 = 2*np.pi*rho/6 * (np.sin(Omega_6 * t) / Omega_6 - np.sin(np.pi * t) / np.pi)
    Phi[:, 2, 0] = np.exp(-kappa * t) * u20 / (2*np.pi)
    
    # ── Level 3 cross-terms (Phi(3,0), Phi(3,1), Phi(3,2)) ──
    # Forcing of level 3: eps*cos(wt/30)*dtheta_3 - eps*cos(wt/6)*dtheta_2/5 + kappa*dR2/5
    
    # Phi(3,2): unit j2, dR=(0,0,2pi,0)
    #   dR2=2*pi*exp(-kt), dtheta_2=0 (only lower levels), dtheta_3=dR2*Pi_2/Pi_3=dR2*6/30=dR2/5
    #   Forcing = eps*cos(wt/30)*dR2/5 - 0 + kappa*dR2/5
    #   = 2*pi*exp(-kt) * rho/5 * [cos(wt/30) + 1]
    Omega_30 = omega / 30  # 2*pi/30
    u32 = 2*np.pi*rho/5 * (np.sin(Omega_30 * t) / Omega_30 + t)
    Phi[:, 3, 2] = np.exp(-kappa * t) * u32 / (2*np.pi)
    
    # Phi(3,1): unit j1, dR=(0,2pi,0,0)
    #   dR1=2*pi*exp(-kt), dR2 via Phi(2,1)
    #   dtheta_2 = dR1*Pi_1/Pi_2 = dR1/3
    #   dtheta_3 = dR1*Pi_1/Pi_3 + dR2*Pi_2/Pi_3 = dR1/15 + dR2/5
    #   At first order in coupling: dR2 ≈ 0 (it's O(rho))
    #   dtheta_3 ≈ dR1/15
    #   Forcing ≈ eps*cos(wt/30)*dR1/15 - eps*cos(wt/6)*dR1/(3*5) + kappa*0
    #   = 2*pi*exp(-kt) * rho * [cos(wt/30)/15 - cos(wt/6)/15]
    #   = 2*pi*exp(-kt) * rho/15 * [cos(wt/30) - cos(wt/6)]
    u31 = 2*np.pi*rho/15 * (np.sin(Omega_30 * t) / Omega_30 - np.sin(Omega_6 * t) / Omega_6)
    Phi[:, 3, 1] = np.exp(-kappa * t) * u31 / (2*np.pi)
    
    # Phi(3,0): unit j0, dR=(2pi,0,0,0)
    #   At first order: dR1≈0, dR2≈0, only dR0 nonzero
    #   dtheta_2 = dR0/6
    #   dtheta_3 = dR0/30
    #   Forcing ≈ eps*cos(wt/30)*dR0/30 - eps*cos(wt/6)*dR0/(5*6) + 0
    #   = 2*pi*exp(-kt) * rho/30 * [cos(wt/30) - cos(wt/6)]
    u30 = 2*np.pi*rho/30 * (np.sin(Omega_30 * t) / Omega_30 - np.sin(Omega_6 * t) / Omega_6)
    Phi[:, 3, 0] = np.exp(-kappa * t) * u30 / (2*np.pi)
    
    return Phi

Phi_ana_full = analytic_Phi_full(t_eval_ci)

# Validate against numerical Phi
print("=" * 70)
print("S6c: FULL ANALYTIC Phi vs NUMERICAL (row 3 at key crossings)")
print("=" * 70)
print(f"{'ci':>6} {'col':>4} {'Phi_num':>12} {'Phi_ana':>12} {'err%':>10}")
print("-" * 50)
for ci_val in [11, 31, 61]:
    idx = np.where(coprime_cis == ci_val)[0][0]
    for col in range(4):
        pn = Phi_at_ci[idx, 3, col]
        pa = Phi_ana_full[idx, 3, col]
        err = (pa/pn - 1)*100 if abs(pn) > 1e-15 else 0
        print(f"{ci_val:6d} {col:4d} {pn:12.4e} {pa:12.4e} {err:9.2f}%")
    print()

# Now compute FULLY ANALYTIC C₀: analytic Phi + first-order R_driv + wrapping
R_sq_ana = np.zeros((N_ci, 4))
for ci_idx in range(N_ci):
    for lev in range(4):
        R_d = R_pert_driven[ci_idx, lev]  # first-order perturbative driven
        phi_row = Phi_ana_full[ci_idx, lev, :]
        sq_sum = 0.0
        for br in all_branches:
            j_arr = np.array(br, dtype=float)
            dR = 2*np.pi * np.dot(phi_row, j_arr)
            R_total = R_d + dR
            R_w = wrap(R_total)
            sq_sum += R_w**2
        R_sq_ana[ci_idx, lev] = sq_sum / len(all_branches)

# Sector C0
cum_sq_a = np.zeros((48, N_ci, 4))
cum_ct_a = np.zeros((48, N_ci), dtype=np.int64)
for j in range(N_ci):
    if j > 0:
        cum_sq_a[:, j, :] = cum_sq_a[:, j-1, :]
        cum_ct_a[:, j] = cum_ct_a[:, j-1]
    s = sector_idx[j]
    cum_sq_a[s, j, :] += R_sq_ana[j, :]
    cum_ct_a[s, j] += 1

j_end = N_ci - 1
lev = 3

g1q = cum_sq_a[QUARK_G1, j_end, lev] / max(cum_ct_a[QUARK_G1, j_end], 1)
g2q = cum_sq_a[QUARK_G2, j_end, lev] / max(cum_ct_a[QUARK_G2, j_end], 1)
C0_Q_ana = np.sqrt(g1q / g2q)

g1l = cum_sq_a[LEPTON_G1, j_end, lev] / max(cum_ct_a[LEPTON_G1, j_end], 1)
g2l = cum_sq_a[LEPTON_G2, j_end, lev] / max(cum_ct_a[LEPTON_G2, j_end], 1)
C0_L_ana = np.sqrt(g1l / g2l)

amp_Q_ana = C0_Q_ana / cp_trans_q
amp_L_ana = C0_L_ana / cp_trans_l

print(f"\nFULLY ANALYTIC C₀ (no ODE integration anywhere):")
print(f"{'':>14} {'C0_ana':>10} {'C0_ODE':>10} {'err%':>8} {'amp':>10}")
print("-" * 55)
print(f"{'QUARK':>14} {C0_Q_ana:10.4f} {ODE_C0_Q:10.4f} {(C0_Q_ana/ODE_C0_Q-1)*100:7.3f}% {amp_Q_ana:10.4f}")
print(f"{'LEPTON':>14} {C0_L_ana:10.4f} {ODE_C0_L:10.4f} {(C0_L_ana/ODE_C0_L-1)*100:7.3f}% {amp_L_ana:10.4f}")
print(f"\nGram targets: ampQ = {TARGET_AMP_Q:.4f}, ampL = {TARGET_AMP_L:.4f}")

S6c: FULL ANALYTIC Phi vs NUMERICAL (row 3 at key crossings)
    ci  col      Phi_num      Phi_ana       err%
--------------------------------------------------
    11    0   6.3687e-03   4.7110e-03    -26.03%
    11    1   1.8571e-02   9.4220e-03    -49.27%
    11    2   9.3451e-02   9.3988e-02      0.57%
    11    3   4.6810e-01   4.6810e-01     -0.00%

    31    0   7.9671e-03   4.4883e-05    -99.44%
    31    1   1.9196e-02   8.9765e-05    -99.53%
    31    2   5.1913e-02   5.1991e-02      0.15%
    31    3   1.1775e-01   1.1775e-01     -0.00%

    61    0   6.6874e-03   5.6624e-06    -99.92%
    61    1   9.0681e-03   1.1325e-05    -99.88%
    61    2   1.2699e-02   1.2710e-02      0.09%
    61    3   1.4855e-02   1.4855e-02     -0.00%


FULLY ANALYTIC C₀ (no ODE integration anywhere):
                   C0_ana     C0_ODE     err%        amp
-------------------------------------------------------
         QUARK     6.3395     6.6067  -4.044%     2.9589
        LEPTON     6.9633   

## Verdict

### What was achieved

NB99 reduces the C₀ computation from a **210-branch ODE integration** to a **state-transition matrix + branch average + wrapping sum**. The full decomposition:

$$C_0^2 = \frac{\langle \text{wrap}(R_3^{driv}(c_1) + 2\pi\sum_k \Phi_{3k}(c_1) j_k)^2 \rangle_j}{\langle \text{wrap}(R_3^{driv}(c_2) + 2\pi\sum_k \Phi_{3k}(c_2) j_k)^2 \rangle_j}$$

where Φ is the cascade Jacobian's state-transition matrix, R_driv is the zero-branch solution, and ⟨⟩_j averages over all 210 branches.

### The amplification mechanism (structural insight)

The 3× amplification (quark) and 5× amplification (lepton) come from **cross-level transient propagation**:
- The branch index j₂ at level 2 shifts θ₃ by 2πj₂·Φ(3,2)·exp(−κt)
- At ci=61 (LEPTON g2), Φ(3,2) ≈ Φ(3,3) — level 2 contributes 25% of R₃ variance
- This cross-level coupling amplifies the sector asymmetry beyond what pure transient weights predict

### Accuracy hierarchy

| Method | QUARK err | LEPTON err | Notes |
|--------|-----------|------------|-------|
| Zeroth-order (decoupled) | ∞ | −2% | Fails for Q because g2 at ci=191 has zero transient |
| First-order perturbative | +2% | +72% | Fails for L because driven R₃ wrong at ci=61 |
| Linearized Jacobian (ODE) | −4% | −2% | Best semi-analytic result |
| Analytic Jacobian | −4% | +18% | Multi-hop Φ(3,0/1) missing |

### Why a closed form doesn't exist

Three obstacles prevent a closed-form C₀:
1. **Wrapping nonlinearity** at early crossings (ci ≤ 31): R² compressed from O(100) to O(1)
2. **Time-dependent Jacobian** coefficients: cos(θ_k^driv(t)) prevents a closed propagator
3. **Branch-dependent wrapping**: the wrapping number n(j) varies with the complete branch vector

### Scorecard: 0 new identities (structural characterization)

NB99 establishes the **mechanism** linking the Gram matrix to C₀ but does not produce a new identity. The Gram-amplification bridge remains structural — validated by the cascade Jacobian anatomy, not by an algebraic derivation.

In [19]:
# ── Scorecard ──
import io

print("NB99 SCORECARD")
print("=" * 65)
print()
print("New identities: 0 (structural characterization)")
print()
print("STRUCTURAL FINDINGS:")
print("  1. C₀ = f(Phi, R_driv, wrap) — decomposed into cascade")
print("     Jacobian state-transition matrix + wrapping sum")
print("  2. Diagonal Phi(k,k) = exp(-kappa*ci) exactly")
print("  3. Single-hop Phi(k,k-1): analytic at 0.1-0.6% accuracy")  
print("  4. Multi-hop Phi(k,m) for m<k-1: requires cascaded")
print("     resonant forcing chain (second-order in coupling)")
print("  5. Wrapping at early crossings (ci<=31) compresses R²")
print("     from O(100) to O(1), preventing closed-form C₀")
print("  6. Cross-level coupling: level 2 contributes 25% of")
print("     R₃ variance at ci=61 (LEPTON g2 crossing)")
print("  7. Best semi-analytic: linearized Jacobian gives")
print("     C₀ to -4% (quark) and -2% (lepton)")
print()
print(f"Running total: 220 predictions/identities, 0 free parameters")

# Save summary
with open('../temp/nb99_results.txt', 'w') as f:
    f.write("NB99: Analytic C₀ Derivation\n")
    f.write("=" * 60 + "\n")
    f.write("RESULT: Structural characterization, 0 new identities\n\n")
    f.write("Semi-analytic C₀ accuracy:\n")
    f.write(f"  Linearized Jacobian: QUARK -4.1%, LEPTON -1.9%\n")
    f.write(f"  Analytic Jacobian:   QUARK -4.0%, LEPTON +17.8%\n\n")
    f.write("Key mechanism: cross-level transient propagation\n")
    f.write("  Phi(3,2) ≈ Phi(3,3) at ci=61 → level 2 drives 25% of R₃\n")
    f.write(f"  Wrapping prevents closed form\n\n")
    f.write(f"Running total: 220 identities (unchanged)\n")
print("\nSaved to temp/nb99_results.txt")

NB99 SCORECARD

New identities: 0 (structural characterization)

STRUCTURAL FINDINGS:
  1. C₀ = f(Phi, R_driv, wrap) — decomposed into cascade
     Jacobian state-transition matrix + wrapping sum
  2. Diagonal Phi(k,k) = exp(-kappa*ci) exactly
  3. Single-hop Phi(k,k-1): analytic at 0.1-0.6% accuracy
  4. Multi-hop Phi(k,m) for m<k-1: requires cascaded
     resonant forcing chain (second-order in coupling)
  5. Wrapping at early crossings (ci<=31) compresses R²
     from O(100) to O(1), preventing closed-form C₀
  6. Cross-level coupling: level 2 contributes 25% of
     R₃ variance at ci=61 (LEPTON g2 crossing)
  7. Best semi-analytic: linearized Jacobian gives
     C₀ to -4% (quark) and -2% (lepton)

Running total: 220 predictions/identities, 0 free parameters

Saved to temp/nb99_results.txt
